# Assignment 3.9: Train Router Small LLM (SFT → DPO) trên Colab

**Chỉ cần upload đúng file notebook này** lên Colab (*File → Upload notebook*). Source code cần thiết đã được nhúng sẵn
(phần *Source code*, mỗi cell `%%writefile` ghi ra 1 file), dữ liệu lấy thẳng từ Hugging Face
[`Tson29/techviet-cs-orchestrator-data`](https://huggingface.co/datasets/Tson29/techviet-cs-orchestrator-data).

1. *Runtime → Change runtime type → **T4 GPU***.
2. Sửa cell **1. Cấu hình** nếu muốn, rồi *Runtime → Run all*. T4 + 1.5B: khoảng 1 giờ; 3B (QLoRA): khoảng 2 giờ.
3. Cuối notebook tải về `a39_router_artifacts.zip` (adapter `sft/`, `dpo/`, dữ liệu, bảng routing eval).
   Giải nén vào thư mục `assignment 39/` trên máy để chạy `python run_chat.py --router dpo`.

`ANTHROPIC_API_KEY` chỉ cần khi `PARAPHRASE > 0` hoặc `EVAL_API = True` (🔑 *Secrets* bên trái, bật *Notebook access*).

> Notebook được sinh bởi `training/build_colab_notebook.py`. Sửa code ở máy rồi build lại, đừng sửa trong các cell `%%writefile`.

In [ ]:
# 1. Cấu hình
BASE_MODEL   = "Qwen/Qwen2.5-1.5B-Instruct"   # hoặc "Qwen/Qwen2.5-3B-Instruct" (giữ QLORA=True), "meta-llama/Llama-3.2-1B-Instruct"
DATA_SOURCE  = "hf+template"  # "hf": chỉ dataset HF | "template": chỉ gen_data | "hf+template": trộn (khuyên dùng)
HF_REPO      = "Tson29/techviet-cs-orchestrator-data"
N_TRAIN      = 1500      # số mẫu SFT sinh từ template (khi DATA_SOURCE có "template")
PARAPHRASE   = 0         # >0: Claude Haiku viết lại N câu cho đa dạng (cần ANTHROPIC_API_KEY, tốn vài cent)
QLORA        = True      # 4-bit, an toàn cho T4 16GB
SFT_EPOCHS   = 2         # DATA_SOURCE="hf" (chỉ vài trăm mẫu): nên để 3
DPO_BETA     = 0.1
DPO_EPOCHS   = 1
ONPOLICY_N   = 400       # số prompt lấy negatives on-policy từ model SFT (0 = chỉ negatives sinh bằng luật)
TEST_LIMIT   = 300       # số mẫu test cho routing eval
EVAL_API     = False     # True: so thêm router Claude API (cần key, tốn tiền)
SAVE_TO_DRIVE = False    # True: chép file kết quả vào Google Drive (MyDrive/a39_router_artifacts.zip), phòng Colab ngắt

import os
os.environ["SMALL_BASE_MODEL"] = BASE_MODEL
os.environ["TOKENIZERS_PARALLELISM"] = "false"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Source code (nhúng sẵn, tự sinh)

In [ ]:
# 2. Tạo thư mục làm việc
import os, pathlib
WORK = pathlib.Path("/content/a39")
for pkg in ['support', 'eval', 'training', 'tests'] + ["data"]:
    (WORK / pkg).mkdir(parents=True, exist_ok=True)
for pkg in ['support', 'eval', 'training', 'tests']:
    (WORK / pkg / "__init__.py").touch()
os.chdir(WORK)
print("Làm việc tại", os.getcwd())

In [ ]:
%%writefile /content/a39/config.py
"""Cấu hình chung cho assignment 3.9. Sửa ở đây thay vì sửa trong code."""
from __future__ import annotations

import os
from pathlib import Path

ROOT = Path(__file__).resolve().parent
DATA_DIR = ROOT / "data"
DOCS_DIR = DATA_DIR / "docs"
RESULTS_DIR = ROOT / "results"
TRACE_DIR = RESULTS_DIR / "traces"
TRAIN_DATA_DIR = ROOT / "training" / "data"
CHECKPOINT_DIR = ROOT / "training" / "checkpoints"


def load_env(path: Path = ROOT / ".env") -> None:
    """Đọc .env đơn giản (KEY=VALUE), không cần python-dotenv."""
    if not path.exists():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))


load_env()

# ---- Model API (Anthropic) ---------------------------------------------------
# Baseline của đề là gpt-oss-120b @ medium reasoning. Bài này dùng Anthropic key, nên
# baseline = Claude Opus 5.5 @ effort "medium" (effort = mức reasoning của Claude).
BASE_MODEL = os.environ.get("BASE_MODEL", "claude-opus-5-5")
BASE_EFFORT = os.environ.get("BASE_EFFORT", "medium")
JUDGE_MODEL = os.environ.get("JUDGE_MODEL", "claude-opus-5-5")
JUDGE_EFFORT = os.environ.get("JUDGE_EFFORT", "low")

# Server-side fallback khi model từ chối (stop_reason = "refusal"). Chỉ Opus/Sonnet/Fable hỗ trợ.
USE_FALLBACKS = os.environ.get("USE_FALLBACKS", "1") == "1"
FALLBACK_MODELS = ("claude-opus-5-5", "claude-opus-5", "claude-sonnet-5-5", "claude-fable-5-1")

# Giá USD / 1M token (Claude API first-party). Cache write = 1.25x input (TTL 5 phút).
# Kiểm tra lại https://www.anthropic.com/pricing trước khi nộp báo cáo.
PRICING = {
    #                     input  output  cache_read
    "claude-opus-5-5":   (4.00, 20.00, 0.20),
    "claude-opus-5":     (5.00, 25.00, 0.50),
    "claude-sonnet-5-5": (2.00, 10.00, 0.20),
    "claude-haiku-5-5":  (0.10, 0.50, 0.01),
    "claude-fable-5-1":  (10.00, 50.00, 0.25),
}
CACHE_WRITE_MULT = 1.25

# Small LLM chạy trên GPU tự host: quy đổi chi phí theo giờ GPU x thời gian xử lý.
GPU_COST_PER_HOUR = float(os.environ.get("GPU_COST_PER_HOUR", "0.35"))   # ~ T4 / L4 thuê theo giờ

# ---- Small LLM ---------------------------------------------------------------
SMALL_BASE_MODEL = os.environ.get("SMALL_BASE_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
SFT_ADAPTER = CHECKPOINT_DIR / "sft"
DPO_ADAPTER = CHECKPOINT_DIR / "dpo"

# ---- RAG ---------------------------------------------------------------------
EMBED_MODEL = os.environ.get("EMBED_MODEL", "intfloat/multilingual-e5-small")
RERANK_MODEL = os.environ.get("RERANK_MODEL", "BAAI/bge-reranker-v2-m3")
CHUNK_CHARS = 700
CHUNK_OVERLAP = 120
RRF_K = 60
TOP_K_RETRIEVE = 12      # số ứng viên mỗi nhánh (BM25 / dense) trước khi fuse
TOP_K_FINAL = 4          # số chunk trả về sau rerank

# Query rewriting trước khi search (support/query_rewriter.py): off | rules | llm
QUERY_REWRITE = os.environ.get("QUERY_REWRITE", "rules")
REWRITE_MODEL = os.environ.get("REWRITE_MODEL", "claude-haiku-5-5")
REWRITE_EFFORT = os.environ.get("REWRITE_EFFORT", "low")

# ---- Serving (FastAPI / Streamlit) -------------------------------------------
API_ROUTER = os.environ.get("API_ROUTER", "api")            # api | rule | small | sft | dpo
API_TOOLS = os.environ.get("API_TOOLS", "local")            # local | mcp
SESSION_TTL_S = int(os.environ.get("SESSION_TTL_S", "3600"))
MAX_SESSIONS = int(os.environ.get("MAX_SESSIONS", "500"))
API_URL = os.environ.get("API_URL", "http://localhost:8000")  # Streamlit gọi FastAPI ở đây

# ---- Agent loop --------------------------------------------------------------
MAX_TOOL_TURNS = 6
AGENT_MAX_TOKENS = 8000
ROUTER_MAX_TOKENS = 4000

AGENTS = ("product", "order", "policy")


In [ ]:
%%writefile /content/a39/support/state.py
"""Conversation state cho State-aware Orchestration.

State được giữ qua nhiều lượt và là input của Router:
- entities     : order_id / phone / product đã biết (slot filling). Regex bắt slot chắc chắn
                 (mã đơn DHxxxx, SĐT 10 số) ngay từ câu khách; router LLM bổ sung phần còn lại.
- pending      : intent còn dở vì thiếu thông tin (vd. khách muốn đổi trả nhưng chưa đưa mã đơn).
                 Lượt sau khách chỉ gõ "DH1005" -> router phải hiểu là tiếp tục đổi trả.
- last_agents  : agent vừa xử lý lượt trước (câu hỏi nối tiếp "còn bảo hành thì sao?").
- history      : vài lượt gần nhất (rút gọn) để router hiểu đại từ "nó", "cái đó".

`router_view()` là đúng chuỗi JSON router nhìn thấy — dùng chung cho API router, Small LLM và
khi sinh dữ liệu train, để train/inference không lệch format.
"""
from __future__ import annotations

import json
import re
import unicodedata
from dataclasses import asdict, dataclass, field

ASK_ORDER_ID = "Bạn vui lòng cho mình mã đơn hàng (dạng DHxxxx) hoặc số điện thoại đặt hàng để mình kiểm tra nhé."

ORDER_RE = re.compile(r"\bD\s?H\s?-?(\d{4})\b", re.I)
PHONE_RE = re.compile(r"\b0\d{9}\b")
PRODUCT_RE = re.compile(r"\bP\d{3}\b", re.I)


def strip_accents(s: str) -> str:
    s = s.replace("đ", "d").replace("Đ", "D")
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")


def extract_slots(text: str) -> dict:
    out = {}
    if m := ORDER_RE.search(text):
        out["order_id"] = f"DH{m.group(1)}"
    if m := PHONE_RE.search(text):
        out["phone"] = m.group(0)
    if m := PRODUCT_RE.search(text):
        out["product_id"] = m.group(0).upper()
    return out


@dataclass
class ConversationState:
    entities: dict = field(default_factory=lambda: {"order_id": "", "phone": "", "product": ""})
    pending_intent: str = ""
    pending_missing: list[str] = field(default_factory=list)
    last_agents: list[str] = field(default_factory=list)
    history: list[dict] = field(default_factory=list)
    turn: int = 0

    def merge(self, slots: dict) -> None:
        for k, v in slots.items():
            if not v:
                continue
            key = "product" if k in ("product", "product_id") else k
            if key in self.entities:
                self.entities[key] = v

    def router_view(self, max_turns: int = 2) -> str:
        view = {
            "entities": {k: v for k, v in self.entities.items() if v},
            "pending_intent": self.pending_intent,
            "pending_missing": self.pending_missing,
            "last_agents": self.last_agents,
            "recent_turns": [{"user": h["user"][:200], "assistant": h["assistant"][:200]}
                             for h in self.history[-max_turns:]],
        }
        return json.dumps(view, ensure_ascii=False)

    def add_turn(self, user: str, assistant: str) -> None:
        self.history.append({"user": user, "assistant": assistant})
        self.turn += 1

    def to_dict(self) -> dict:
        return asdict(self)

    @classmethod
    def from_view(cls, view: str | dict) -> "ConversationState":
        """Dựng lại state từ router_view() (dùng cho routing eval trên tập test đã sinh sẵn)."""
        v = json.loads(view) if isinstance(view, str) else view
        st = cls(pending_intent=v.get("pending_intent", ""), pending_missing=list(v.get("pending_missing", [])),
                 last_agents=list(v.get("last_agents", [])), history=list(v.get("recent_turns", [])))
        st.merge(v.get("entities", {}))
        st.turn = len(st.history)
        return st


In [ ]:
%%writefile /content/a39/support/tracing.py
"""Observability: tracing dạng span lồng nhau + tính token / chi phí / latency.

    with tracer.span("agent.order", kind="agent", input=...) as sp:
        ...
        sp.set(output=..., tokens=...)

- Mỗi request của khách = 1 trace (trace_id). Span con: router, a2a.send, agent, llm, tool, rag.
- Span "llm" mang model, effort, input/output/cache tokens, cost_usd. Tổng của trace = cộng các span llm.
- Ghi JSONL (results/traces/<run>.jsonl), mỗi dòng là 1 span đã đóng -> dashboard.py đọc lại.
- Thread-safe đủ dùng cho eval tuần tự; parent span theo contextvars nên A2A in-process vẫn lồng đúng.
"""
from __future__ import annotations

import json
import logging
import threading
import time
import uuid
from collections import deque
from contextlib import contextmanager
from contextvars import ContextVar
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import config

_current: ContextVar["Span | None"] = ContextVar("current_span", default=None)
_current_tracer: ContextVar["Tracer | None"] = ContextVar("current_tracer", default=None)
_log = logging.getLogger("support.tracing")


def llm_cost(model: str, usage: dict) -> float:
    """usage: input_tokens, output_tokens, cache_read_input_tokens, cache_creation_input_tokens."""
    p_in, p_out, p_cache = config.PRICING.get(model, config.PRICING[config.BASE_MODEL])
    return (usage.get("input_tokens", 0) * p_in
            + usage.get("cache_creation_input_tokens", 0) * p_in * config.CACHE_WRITE_MULT
            + usage.get("cache_read_input_tokens", 0) * p_cache
            + usage.get("output_tokens", 0) * p_out) / 1_000_000


def gpu_cost(seconds: float) -> float:
    return seconds / 3600 * config.GPU_COST_PER_HOUR


@dataclass
class Span:
    name: str
    kind: str
    trace_id: str
    parent_id: str | None
    span_id: str = field(default_factory=lambda: uuid.uuid4().hex[:16])
    start: float = field(default_factory=time.time)
    attrs: dict[str, Any] = field(default_factory=dict)
    status: str = "ok"
    latency_ms: float = 0.0

    def set(self, **kw) -> None:
        self.attrs.update(kw)

    def to_dict(self) -> dict:
        return {"trace_id": self.trace_id, "span_id": self.span_id, "parent_id": self.parent_id,
                "name": self.name, "kind": self.kind, "start": self.start,
                "latency_ms": round(self.latency_ms, 1), "status": self.status, "attrs": self.attrs}


class Tracer:
    def __init__(self, run_name: str | None = None, out_dir: Path = config.TRACE_DIR, enabled: bool = True,
                 max_spans: int | None = None):
        """max_spans: chỉ giữ N span gần nhất trong RAM (API server chạy lâu); file JSONL vẫn ghi đủ."""
        self.enabled = enabled
        self.spans: list[dict] | deque = deque(maxlen=max_spans) if max_spans else []
        self._lock = threading.Lock()
        self.path = None
        if enabled:
            out_dir.mkdir(parents=True, exist_ok=True)
            self.path = out_dir / f"{run_name or time.strftime('%Y%m%d-%H%M%S')}.jsonl"

    @contextmanager
    def span(self, name: str, kind: str = "internal", trace_id: str | None = None, **attrs):
        parent = _current.get()
        sp = Span(name=name, kind=kind, trace_id=trace_id or (parent.trace_id if parent else uuid.uuid4().hex[:16]),
                  parent_id=parent.span_id if parent else None, attrs=_safe(attrs))
        token = _current.set(sp)
        ttoken = _current_tracer.set(self)
        t0 = time.perf_counter()
        try:
            yield sp
        except Exception as e:
            sp.status = "error"
            sp.attrs["error"] = f"{type(e).__name__}: {e}"
            _log.warning("span failed", extra={"span": name, "kind": kind, "error": sp.attrs["error"]})
            raise
        finally:
            sp.latency_ms = (time.perf_counter() - t0) * 1000
            if _log.isEnabledFor(logging.DEBUG):
                _log.debug("span closed", extra={"span": name, "kind": kind,
                                                 "latency_ms": round(sp.latency_ms, 1), "status": sp.status})
            _current.reset(token)
            _current_tracer.reset(ttoken)
            self._emit(sp)

    def _emit(self, sp: Span) -> None:
        d = sp.to_dict()
        d["attrs"] = _safe(d["attrs"])
        line = json.dumps(d, ensure_ascii=False) + "\n"
        with self._lock:                                   # API server: nhiều request ghi cùng file
            self.spans.append(d)
            if self.path:
                with self.path.open("a", encoding="utf-8") as f:
                    f.write(line)

    # ---- tổng hợp cho 1 trace -------------------------------------------------
    def summary(self, trace_id: str) -> dict:
        with self._lock:
            spans = [s for s in self.spans if s["trace_id"] == trace_id]
        llm = [s for s in spans if s["kind"] == "llm"]
        tools = [s for s in spans if s["kind"] == "tool"]
        root = next((s for s in spans if s["parent_id"] is None), None)
        return {
            "trace_id": trace_id,
            "latency_ms": root["latency_ms"] if root else sum(s["latency_ms"] for s in spans),
            "llm_calls": len(llm),
            "input_tokens": sum(s["attrs"].get("input_tokens", 0) for s in llm),
            "output_tokens": sum(s["attrs"].get("output_tokens", 0) for s in llm),
            "cache_read_tokens": sum(s["attrs"].get("cache_read_input_tokens", 0) for s in llm),
            "cost_usd": round(sum(s["attrs"].get("cost_usd", 0.0) for s in spans), 6),
            "tool_calls": [{"name": s["attrs"].get("tool"), "args": s["attrs"].get("args")} for s in tools],
            "errors": [s["name"] for s in spans if s["status"] == "error"],
        }


def current_trace_id() -> str | None:
    sp = _current.get()
    return sp.trace_id if sp else None


def current_span() -> Span | None:
    return _current.get()


def current_tracer() -> "Tracer | None":
    """Tracer của span đang mở: code sâu bên dưới (vd. query rewriter trong tool) ghi span vào đúng trace."""
    return _current_tracer.get()


def _safe(obj: Any, limit: int = 4000) -> Any:
    """Cắt bớt chuỗi dài để file trace không phình."""
    if isinstance(obj, str):
        return obj if len(obj) <= limit else obj[:limit] + f"...(+{len(obj) - limit})"
    if isinstance(obj, dict):
        return {k: _safe(v, limit) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_safe(v, limit) for v in obj]
    if isinstance(obj, (int, float, bool)) or obj is None:
        return obj
    return _safe(str(obj), limit)


NULL_TRACER = Tracer(enabled=False, max_spans=1000)   # không ghi file; giới hạn RAM khi chạy lâu


In [ ]:
%%writefile /content/a39/support/router.py
"""Router / Planner của Orchestrator: quyết định intent, agent nào xử lý, theo thứ tự nào, thiếu gì.

Output (RoutingDecision, JSON):
    {"intent": "...", "agents": ["order","policy"], "plan": [{"agent":"order","task":"..."}, ...],
     "missing_info": ["order_id"], "entities": {"order_id":"", "phone":"", "product":""}, "reply": ""}

Ba cài đặt cùng interface `decide(state, user_msg) -> dict`:
- APIRouter      : Claude + structured output (JSON Schema), baseline "API LLM".
- SmallLLMRouter : model 1B–3B (HF transformers), base / + LoRA SFT / + LoRA DPO.
- RuleRouter     : từ khoá, baseline rẻ nhất, dùng để so sánh và làm fallback khi JSON hỏng.

ROUTER_SYSTEM + build_router_user() là prompt DUY NHẤT, dùng cho cả train lẫn inference.
"""
from __future__ import annotations

import json
import re
import time

import config
from support.state import ConversationState
from support.tracing import NULL_TRACER, Tracer, gpu_cost

INTENTS = ["product_inquiry", "order_status", "cancel_order", "return_request", "warranty_claim",
           "policy_question", "multi", "chitchat", "out_of_scope"]
NEEDS_ORDER = {"order_status", "cancel_order", "return_request", "warranty_claim"}

ROUTER_SYSTEM = """Bạn là Router của hệ thống chăm sóc khách hàng TechViet. Đọc STATE và tin nhắn khách, trả về JSON quyết định điều phối.

Agents:
- product: tư vấn, tìm, so sánh sản phẩm, thông số, giá, tồn kho, hướng dẫn sử dụng.
- order: tra cứu trạng thái đơn, vận chuyển, huỷ đơn.
- policy: chính sách đổi trả/bảo hành/vận chuyển/thanh toán; kiểm tra điều kiện và tạo phiếu đổi trả, bảo hành.

Intent -> agents:
- product_inquiry -> [product]
- order_status, cancel_order -> [order]
- return_request, warranty_claim, policy_question -> [policy]
- multi: câu hỏi cần nhiều agent -> liệt kê đủ, theo thứ tự cần chạy
- chitchat (chào hỏi, cảm ơn) / out_of_scope (không liên quan cửa hàng) -> agents [] và viết reply ngắn

Quy tắc:
1. order_status, cancel_order, return_request, warranty_claim cần order_id. Nếu STATE và tin nhắn đều không có order_id (order_status chấp nhận phone thay order_id) -> missing_info ["order_id"], agents [], plan [], reply hỏi khách mã đơn.
2. Nếu STATE có pending_intent và tin nhắn bổ sung thông tin còn thiếu -> tiếp tục pending_intent.
3. Câu nối tiếp ngắn ("còn bảo hành thì sao?", "nó còn hàng không?") hiểu theo recent_turns và entities.
4. plan: mỗi bước một agent, task là chỉ dẫn cụ thể (kèm mã đơn / sản phẩm nếu có).
5. entities: điền order_id, phone, product (tên hoặc mã) biết được từ STATE + tin nhắn; không biết thì "".
6. reply chỉ dùng khi agents rỗng; ngược lại để "".
Chỉ trả về JSON."""

ROUTER_SCHEMA = {
    "type": "object",
    "properties": {
        "intent": {"type": "string", "enum": INTENTS},
        "agents": {"type": "array", "items": {"type": "string", "enum": list(config.AGENTS)}},
        "plan": {"type": "array", "items": {
            "type": "object",
            "properties": {"agent": {"type": "string", "enum": list(config.AGENTS)}, "task": {"type": "string"}},
            "required": ["agent", "task"], "additionalProperties": False}},
        "missing_info": {"type": "array", "items": {"type": "string", "enum": ["order_id"]}},
        "entities": {"type": "object",
                     "properties": {"order_id": {"type": "string"}, "phone": {"type": "string"},
                                    "product": {"type": "string"}},
                     "required": ["order_id", "phone", "product"], "additionalProperties": False},
        "reply": {"type": "string"},
    },
    "required": ["intent", "agents", "plan", "missing_info", "entities", "reply"],
    "additionalProperties": False,
}


def build_router_user(state_view: str, user_msg: str) -> str:
    return f"STATE: {state_view}\nKHÁCH: {user_msg}"


def normalize_decision(d: dict) -> dict:
    """Đưa output (nhất là của small LLM) về đúng hình dạng; agents suy ra từ plan nếu thiếu."""
    d = dict(d) if isinstance(d, dict) else {}
    intent = d.get("intent") if d.get("intent") in INTENTS else "out_of_scope"
    plan = [s for s in d.get("plan") or [] if isinstance(s, dict) and s.get("agent") in config.AGENTS]
    agents = [a for a in d.get("agents") or [] if a in config.AGENTS]
    for s in plan:
        if s["agent"] not in agents:
            agents.append(s["agent"])
    if agents and not plan:
        plan = [{"agent": a, "task": ""} for a in agents]
    ents = d.get("entities") if isinstance(d.get("entities"), dict) else {}
    missing = [m for m in d.get("missing_info") or [] if m == "order_id"]
    if missing:
        agents, plan = [], []
    return {"intent": intent, "agents": agents, "plan": plan, "missing_info": missing,
            "entities": {k: str(ents.get(k) or "") for k in ("order_id", "phone", "product")},
            "reply": str(d.get("reply") or "")}


def parse_json_obj(text: str) -> dict | None:
    text = text.strip()
    text = re.sub(r"^```(?:json)?|```$", "", text, flags=re.M).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.S)
        if m:
            try:
                return json.loads(m.group(0))
            except json.JSONDecodeError:
                return None
    return None


# --------------------------------------------------------------------------- #
class APIRouter:
    def __init__(self, model: str = config.BASE_MODEL, effort: str = config.BASE_EFFORT, tracer: Tracer = NULL_TRACER):
        from support.llm import Claude

        self.llm = Claude(model=model, effort=effort, max_tokens=config.ROUTER_MAX_TOKENS, tracer=tracer, name="router")
        self.tracer = tracer
        self.label = f"api:{model}@{effort}"

    def decide(self, state: ConversationState, user_msg: str) -> dict:
        user = build_router_user(state.router_view(), user_msg)
        with self.tracer.span("router", kind="router", router=self.label) as sp:
            d = normalize_decision(self.llm.structured(ROUTER_SYSTEM, user, ROUTER_SCHEMA))
            sp.set(decision=d)
            return d


class SmallLLMRouter:
    """Qwen2.5-1.5B-Instruct (mặc định) + LoRA adapter (SFT hoặc DPO). Cần GPU để chạy nhanh;
    trên CPU vẫn chạy được nhưng chậm (vài giây / quyết định)."""

    def __init__(self, base_model: str = config.SMALL_BASE_MODEL, adapter: str | None = None,
                 tracer: Tracer = NULL_TRACER, load_4bit: bool = False, label: str | None = None):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.torch = torch
        self.tok = AutoTokenizer.from_pretrained(adapter or base_model)
        # T4 / P100 (Colab, Kaggle) không có bf16 native -> fp16; CPU -> fp32
        if torch.cuda.is_available():
            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        else:
            dtype = torch.float32
        kw = {"torch_dtype": dtype}
        if load_4bit:
            from transformers import BitsAndBytesConfig

            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                            bnb_4bit_compute_dtype=dtype)
        if torch.cuda.is_available():
            kw["device_map"] = "auto"
        model = AutoModelForCausalLM.from_pretrained(base_model, **kw)
        if adapter:
            from peft import PeftModel

            model = PeftModel.from_pretrained(model, adapter)
        self.model = model.eval()
        self.tracer = tracer
        self.label = label or f"small:{base_model.split('/')[-1]}" + (f"+{adapter.rstrip('/').split('/')[-1]}" if adapter else "")
        self.fallback = RuleRouter()

    def generate(self, user: str, max_new_tokens: int = 320) -> tuple[str, dict]:
        msgs = [{"role": "system", "content": ROUTER_SYSTEM}, {"role": "user", "content": user}]
        enc = self.tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)
        ids = enc["input_ids"].to(self.model.device)
        t0 = time.perf_counter()
        with self.torch.no_grad():
            out = self.model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                                      pad_token_id=self.tok.eos_token_id)
        secs = time.perf_counter() - t0
        new = out[0, ids.shape[1]:]
        return self.tok.decode(new, skip_special_tokens=True), {
            "input_tokens": int(ids.shape[1]), "output_tokens": int(new.shape[0]),
            "gen_seconds": round(secs, 3), "cost_usd": gpu_cost(secs)}

    def decide(self, state: ConversationState, user_msg: str) -> dict:
        user = build_router_user(state.router_view(), user_msg)
        with self.tracer.span("router", kind="router", router=self.label) as sp:
            text, usage = self.generate(user)
            raw = parse_json_obj(text)
            sp.set(raw_output=text, **usage, json_valid=raw is not None)
            d = normalize_decision(raw) if raw is not None else self.fallback.decide(state, user_msg)
            sp.set(decision=d)
            return d


class RuleRouter:
    """Baseline từ khoá. Cố tình đơn giản: cho thấy vì sao cần router học được ngữ cảnh."""

    label = "rule"
    KW = {
        "cancel_order": ["huỷ đơn", "hủy đơn", "huỷ", "hủy", "cancel"],
        "warranty_claim": ["bảo hành", "hỏng", "lỗi", "không lên", "điểm chết", "sửa"],
        "return_request": ["đổi trả", "trả hàng", "hoàn tiền", "đổi ý", "trả lại", "đổi sang"],
        "order_status": ["đơn hàng", "giao", "vận đơn", "đơn của", "ship", "khi nào nhận", "tới chưa"],
        "policy_question": ["chính sách", "trả góp", "phí vận chuyển", "thanh toán", "vat", "cod"],
        "product_inquiry": ["tư vấn", "laptop", "tai nghe", "điện thoại", "giá", "còn hàng", "so sánh", "pin",
                            "đồng hồ", "màn hình", "chuột", "bàn phím", "loa", "máy tính bảng", "sạc", "gợi ý"],
        "chitchat": ["xin chào", "chào", "cảm ơn", "cám ơn", "hello", "hi "],
    }
    AGENT = {"product_inquiry": "product", "order_status": "order", "cancel_order": "order",
             "return_request": "policy", "warranty_claim": "policy", "policy_question": "policy"}

    def __init__(self, tracer: Tracer = NULL_TRACER):
        self.tracer = tracer

    def decide(self, state: ConversationState, user_msg: str) -> dict:
        low = user_msg.lower()
        hits = [i for i, kws in self.KW.items() if any(k in low for k in kws)]
        has_order = bool(state.entities.get("order_id") or state.entities.get("phone"))
        if not hits and state.pending_intent:
            hits = [state.pending_intent]
        task_hits = [h for h in hits if h in self.AGENT]
        if not task_hits:
            intent = "chitchat" if "chitchat" in hits else "out_of_scope"
            reply = "Chào bạn, TechViet có thể giúp gì cho bạn?" if intent == "chitchat" else \
                "Xin lỗi, mình chỉ hỗ trợ các vấn đề về sản phẩm, đơn hàng, đổi trả và bảo hành của TechViet."
            return normalize_decision({"intent": intent, "reply": reply, "entities": state.entities})
        agents = list(dict.fromkeys(self.AGENT[h] for h in task_hits))
        intent = task_hits[0] if len(agents) == 1 else "multi"
        if any(h in NEEDS_ORDER for h in task_hits) and not has_order:
            return normalize_decision({"intent": intent, "missing_info": ["order_id"], "entities": state.entities,
                                       "reply": "Bạn vui lòng cho mình mã đơn hàng (dạng DHxxxx) nhé."})
        return normalize_decision({"intent": intent, "agents": agents, "entities": state.entities,
                                   "plan": [{"agent": a, "task": user_msg} for a in agents]})


def make_router(kind: str, tracer: Tracer = NULL_TRACER, model: str | None = None, effort: str | None = None):
    """kind: api | rule | small | sft | dpo"""
    if kind == "api":
        return APIRouter(model or config.BASE_MODEL, effort or config.BASE_EFFORT, tracer)
    if kind == "rule":
        return RuleRouter(tracer)
    adapter = {"small": None, "sft": str(config.SFT_ADAPTER), "dpo": str(config.DPO_ADAPTER)}[kind]
    return SmallLLMRouter(adapter=adapter, tracer=tracer, label=f"small-{kind}")


In [ ]:
%%writefile /content/a39/support/llm.py
"""Wrapper gọi Claude dùng chung cho Router (API) / Product / Order / Policy agent / Judge.

- Mỗi lời gọi = 1 span "llm" (model, effort, tokens, cost, latency).
- Prompt caching: system prompt + danh sách tool được đánh cache_control -> các lượt sau trong
  vòng tool-use và các request sau đọc từ cache (rẻ hơn ~20 lần với Opus 5.5).
- Refusal: bật server-side fallback (beta `server-side-fallback-2026-07-01`, fallbacks="default")
  cho các model hỗ trợ; vẫn kiểm tra stop_reason == "refusal" trước khi đọc content.
- Không gửi temperature (API version mới trả 400); độ sâu suy luận điều khiển bằng output_config.effort.
"""
from __future__ import annotations

import json
import os
import time
from contextlib import contextmanager
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable

import config
from support.tracing import NULL_TRACER, Tracer, llm_cost


class RefusalError(RuntimeError):
    pass


class BudgetExceeded(BaseException):
    """Hết ngân sách API. Kế thừa BaseException để các `except Exception` (A2A, eval) không nuốt mất -> dừng hẳn."""


# ---- Sổ chi tiêu dùng chung mọi process (bật khi đặt API_BUDGET_USD) -------------------------------------
def _ledger_path() -> Path:
    return Path(os.environ.get("SPEND_LEDGER") or config.RESULTS_DIR / "spend.json")


@contextmanager
def _locked():
    path = _ledger_path()
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(str(path) + ".lock", "w") as lk:
        try:
            import fcntl

            fcntl.flock(lk, fcntl.LOCK_EX)
        except ImportError:                       # Windows: chạy tuần tự là đủ
            pass
        yield path


def spent() -> dict:
    p = _ledger_path()
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else {"total_usd": 0.0, "calls": 0, "by_model": {}}


def _budget() -> float:
    return float(os.environ.get("API_BUDGET_USD", "0") or 0)


def check_budget() -> None:
    b = _budget()
    if b and spent()["total_usd"] >= b:
        raise BudgetExceeded(f"Đã tiêu {spent()['total_usd']:.4f}$ >= ngân sách {b}$ (API_BUDGET_USD)")


def record_spend(model: str, usd: float) -> None:
    if not _budget():
        return
    with _locked() as p:
        d = json.loads(p.read_text(encoding="utf-8")) if p.exists() else {"total_usd": 0.0, "calls": 0, "by_model": {}}
        d["total_usd"] = round(d["total_usd"] + usd, 6)
        d["calls"] += 1
        d["by_model"][model] = round(d["by_model"].get(model, 0.0) + usd, 6)
        p.write_text(json.dumps(d, indent=1), encoding="utf-8")


def _usage_dict(u) -> dict:
    return {"input_tokens": u.input_tokens or 0, "output_tokens": u.output_tokens or 0,
            "cache_read_input_tokens": getattr(u, "cache_read_input_tokens", 0) or 0,
            "cache_creation_input_tokens": getattr(u, "cache_creation_input_tokens", 0) or 0}


def text_of(resp) -> str:
    return "".join(b.text for b in resp.content if b.type == "text")


@dataclass
class Claude:
    model: str = config.BASE_MODEL
    effort: str = config.BASE_EFFORT
    max_tokens: int = config.AGENT_MAX_TOKENS
    tracer: Tracer = field(default=NULL_TRACER)
    name: str = "llm"

    def __post_init__(self):
        import anthropic

        self.client = anthropic.Anthropic()

    @property
    def label(self) -> str:
        return f"{self.model}@{self.effort}"

    # ------------------------------------------------------------------ #
    def create(self, system: str, messages: list[dict], tools: list[dict] | None = None,
               schema: dict | None = None, tool_choice: dict | None = None):
        output_config: dict = {"effort": self.effort}
        if schema:
            output_config["format"] = {"type": "json_schema", "schema": schema}
        kw: dict = {
            "model": self.model, "max_tokens": self.max_tokens, "messages": messages,
            "system": [{"type": "text", "text": system, "cache_control": {"type": "ephemeral"}}],
            "output_config": output_config,
        }
        if tools:
            tools = [dict(t) for t in tools]
            tools[-1]["cache_control"] = {"type": "ephemeral"}       # cache toàn bộ khối tools
            kw["tools"] = tools
            kw["tool_choice"] = tool_choice or {"type": "auto"}

        check_budget()
        with self.tracer.span(f"llm.{self.name}", kind="llm", model=self.model, effort=self.effort) as sp:
            t0 = time.perf_counter()
            if config.USE_FALLBACKS and self.model in config.FALLBACK_MODELS:
                resp = self.client.beta.messages.create(
                    betas=["server-side-fallback-2026-07-01"], fallbacks="default", **kw)
            else:
                resp = self.client.messages.create(**kw)
            usage = _usage_dict(resp.usage)
            served_by = getattr(resp, "model", self.model) or self.model
            cost = llm_cost(served_by, usage)
            record_spend(served_by, cost)
            sp.set(**usage, served_by=served_by, stop_reason=resp.stop_reason,
                   cost_usd=cost, api_latency_ms=round((time.perf_counter() - t0) * 1000, 1))
            if resp.stop_reason == "refusal":
                raise RefusalError(f"model từ chối: {getattr(resp, 'stop_details', None)}")
            return resp

    # ------------------------------------------------------------------ #
    def structured(self, system: str, user: str | list[dict], schema: dict) -> dict:
        messages = [{"role": "user", "content": user}] if isinstance(user, str) else user
        resp = self.create(system, messages, schema=schema)
        if resp.stop_reason == "max_tokens":
            raise RuntimeError("structured output bị cắt do max_tokens")
        return json.loads(text_of(resp))

    def text(self, system: str, user: str) -> str:
        return text_of(self.create(system, [{"role": "user", "content": user}]))

    def tool_loop(self, system: str, messages: list[dict], tools: list[dict],
                  execute: Callable[[str, dict], dict], max_turns: int = config.MAX_TOOL_TURNS) -> tuple[str, list[dict]]:
        """Vòng tool use thủ công. Trả về (câu trả lời cuối, danh sách tool call đã thực hiện)."""
        messages = list(messages)
        calls: list[dict] = []
        for turn in range(max_turns):
            last = turn == max_turns - 1
            resp = self.create(system, messages, tools=tools,
                               tool_choice={"type": "none"} if last else {"type": "auto"})
            # giữ nguyên các block (kể cả thinking) để gửi lại đúng thứ tự; bỏ block "fallback" (metadata)
            messages.append({"role": "assistant",
                             "content": [b.to_dict() for b in resp.content if b.type != "fallback"]})
            if resp.stop_reason != "tool_use":
                return text_of(resp), calls
            results = []
            for b in resp.content:
                if b.type != "tool_use":
                    continue
                out = execute(b.name, dict(b.input))
                calls.append({"name": b.name, "args": dict(b.input)})
                results.append({"type": "tool_result", "tool_use_id": b.id,
                                "content": json.dumps(out, ensure_ascii=False),
                                "is_error": isinstance(out, dict) and "error" in out})
            messages.append({"role": "user", "content": results})
        return "Xin lỗi, tôi chưa xử lý xong yêu cầu này.", calls


In [ ]:
%%writefile /content/a39/eval/metrics.py
"""Metrics không cần LLM: Routing accuracy và Tool-call F1.

Routing đúng (mỗi lượt) khi: tập agents dự đoán == tập agents kỳ vọng VÀ (có hỏi lại thiếu thông tin) khớp.
Thứ tự agents không tính (thứ tự chỉ ảnh hưởng chất lượng câu trả lời, đã có Judge đo).

Tool-call F1 (mỗi kịch bản, gộp mọi lượt):
- Một call dự đoán khớp call kỳ vọng khi cùng tên tool VÀ mọi tham số được nêu trong expected đều bằng nhau
  (so sánh không phân biệt hoa thường; list so như tập). Tham số không nêu -> không xét
  (vd. query của search_* là câu tự do nên expected để {}).
- Ghép tham lam 1-1: TP = số cặp khớp; FP = call thừa (gọi sai / gọi lặp / gọi không cần);
  FN = call kỳ vọng bị thiếu. Precision phạt agent "gọi lung tung" (tốn latency + tiền), recall phạt agent bỏ bước.
- Kịch bản không cần tool và agent không gọi tool -> P = R = F1 = 1.
Báo cáo micro-F1 (cộng TP/FP/FN toàn bộ) và macro-F1 (trung bình theo kịch bản).
"""
from __future__ import annotations


def routing_correct(pred: dict, gold_agents: list[str], gold_missing: bool) -> bool:
    return set(pred.get("agents", [])) == set(gold_agents) and bool(pred.get("missing_info")) == gold_missing


def _norm(v):
    if isinstance(v, (list, tuple)):
        return tuple(sorted(_norm(x) for x in v))
    if isinstance(v, str):
        return v.strip().upper()
    return v


def call_matches(pred: dict, exp: dict) -> bool:
    if pred.get("name") != exp["name"]:
        return False
    args = pred.get("args") or {}
    return all(_norm(args.get(k)) == _norm(v) for k, v in exp.get("args", {}).items())


def tool_f1(pred_calls: list[dict], exp_calls: list[dict]) -> dict:
    used = [False] * len(pred_calls)
    tp = 0
    for e in exp_calls:
        for i, p in enumerate(pred_calls):
            if not used[i] and call_matches(p, e):
                used[i] = True
                tp += 1
                break
    fp, fn = len(pred_calls) - tp, len(exp_calls) - tp
    if not pred_calls and not exp_calls:
        return {"tp": 0, "fp": 0, "fn": 0, "precision": 1.0, "recall": 1.0, "f1": 1.0}
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": round(p, 4), "recall": round(r, 4), "f1": round(f1, 4)}


def micro_f1(rows: list[dict]) -> dict:
    tp = sum(r["tp"] for r in rows)
    fp = sum(r["fp"] for r in rows)
    fn = sum(r["fn"] for r in rows)
    p = tp / (tp + fp) if tp + fp else 1.0
    r = tp / (tp + fn) if tp + fn else 1.0
    return {"precision": round(p, 4), "recall": round(r, 4), "f1": round(2 * p * r / (p + r) if p + r else 0.0, 4)}


def percentile(xs: list[float], q: float) -> float:
    if not xs:
        return 0.0
    xs = sorted(xs)
    k = (len(xs) - 1) * q
    lo, hi = int(k), min(int(k) + 1, len(xs) - 1)
    return xs[lo] + (xs[hi] - xs[lo]) * (k - lo)


In [ ]:
%%writefile /content/a39/eval/routing_eval.py
"""Routing-only eval: so sánh API LLM vs Small LLM (base) vs SFT vs DPO (vs rule) ở bước điều phối.

Không chạy agent nên rẻ và nhanh: chỉ đo quyết định của Router trên
  - eval set (data/eval_set.json): mỗi lượt với STATE "chuẩn" (mô phỏng các lượt trước đúng như kỳ vọng)
  - tập test sinh sẵn (training/data/sft_test.jsonl), có đủ nhóm state-aware.

Metrics: routing accuracy (tập agents + có hỏi lại), intent accuracy, JSON valid rate (small LLM),
latency trung bình / p95, chi phí / 1000 quyết định (API: theo token; small: theo giờ GPU).

    python -m eval.routing_eval --routers rule api                    # không cần GPU
    python -m eval.routing_eval --routers small sft dpo --test-limit 300   # trên máy có GPU
    python -m eval.routing_eval --routers api --api-model claude-haiku-5-5 --api-effort low
"""
from __future__ import annotations

import argparse
import copy
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import config  # noqa: E402
from eval.metrics import percentile, routing_correct  # noqa: E402
from support.router import make_router  # noqa: E402
from support.state import ASK_ORDER_ID, ConversationState, extract_slots  # noqa: E402
from support.tracing import Tracer  # noqa: E402


def gold_turns(scn: dict):
    """Sinh (state, user_msg, gold) cho từng lượt, giả định các lượt trước được xử lý đúng."""
    st = ConversationState()
    for turn in scn["turns"]:
        st.merge(extract_slots(turn["user"]))
        yield copy.deepcopy(st), turn["user"], {"intent": turn["intent"], "agents": turn["agents"], "missing": turn["missing"]}
        if turn["missing"]:
            st.pending_intent, st.pending_missing, reply = turn["intent"], ["order_id"], ASK_ORDER_ID
        else:
            st.pending_intent, st.pending_missing, reply = "", [], "(trợ lý đã trả lời yêu cầu)"
            st.last_agents = turn["agents"] or st.last_agents
        st.add_turn(turn["user"], reply)


def load_cases(test_limit: int) -> list[dict]:
    cases = []
    for scn in json.loads((config.DATA_DIR / "eval_set.json").read_text(encoding="utf-8")):
        for i, (st, msg, gold) in enumerate(gold_turns(scn)):
            cases.append({"src": "eval_set", "id": f"{scn['id']}.{i + 1}", "kind": scn["category"],
                          "state": st, "msg": msg, "gold": gold})
    test = config.TRAIN_DATA_DIR / "sft_test.jsonl"
    if test.exists() and test_limit:
        for j, line in enumerate(test.open(encoding="utf-8")):
            if j >= test_limit:
                break
            r = json.loads(line)
            t = r["target"]
            cases.append({"src": "test", "id": f"T{j}", "kind": r["kind"], "state": ConversationState.from_view(r["state"]),
                          "msg": r["user_message"],
                          "gold": {"intent": t["intent"], "agents": t["agents"], "missing": bool(t["missing_info"])}})
    return cases


def evaluate(router, tracer: Tracer, cases: list[dict]) -> dict:
    rows = []
    for c in cases:
        with tracer.span("routing_case", kind="eval", case=c["id"]) as root:
            try:
                d = router.decide(c["state"], c["msg"])
                err = None
            except Exception as e:
                d, err = {"intent": "", "agents": [], "missing_info": []}, f"{type(e).__name__}: {e}"
        s = tracer.summary(root.trace_id)
        rspan = next((x for x in tracer.spans if x["trace_id"] == root.trace_id and x["kind"] == "router"), {"attrs": {}})
        rows.append({"id": c["id"], "src": c["src"], "kind": c["kind"], "msg": c["msg"], "gold": c["gold"], "pred": d,
                     "correct": routing_correct(d, c["gold"]["agents"], c["gold"]["missing"]),
                     "intent_ok": d.get("intent") == c["gold"]["intent"],
                     "json_valid": rspan["attrs"].get("json_valid", err is None),
                     "latency_ms": s["latency_ms"], "cost_usd": s["cost_usd"], "error": err})
    return {"rows": rows, "summary": summarize(rows)}


def summarize(rows: list[dict]) -> dict:
    def acc(rs, key="correct"):
        return round(sum(r[key] for r in rs) / len(rs), 4) if rs else None

    lat = [r["latency_ms"] for r in rows]
    by_kind = {}
    for k in sorted({r["kind"] for r in rows}):
        rs = [r for r in rows if r["kind"] == k]
        by_kind[k] = {"n": len(rs), "acc": acc(rs)}
    return {
        "n": len(rows),
        "routing_acc": acc(rows), "intent_acc": acc(rows, "intent_ok"), "json_valid": acc(rows, "json_valid"),
        "routing_acc_eval_set": acc([r for r in rows if r["src"] == "eval_set"]),
        "routing_acc_test": acc([r for r in rows if r["src"] == "test"]),
        "latency_ms_mean": round(sum(lat) / len(lat), 1) if lat else 0,
        "latency_ms_p95": round(percentile(lat, 0.95), 1),
        "cost_per_1k_usd": round(sum(r["cost_usd"] for r in rows) / len(rows) * 1000, 4) if rows else 0,
        "errors": sum(1 for r in rows if r["error"]),
        "by_kind": by_kind,
    }


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--routers", nargs="+", default=["rule", "api"], help="rule api small sft dpo")
    ap.add_argument("--api-model", default=config.BASE_MODEL)
    ap.add_argument("--api-effort", default=config.BASE_EFFORT)
    ap.add_argument("--test-limit", type=int, default=150, help="số mẫu lấy từ sft_test.jsonl (0 = bỏ)")
    a = ap.parse_args()

    cases = load_cases(a.test_limit)
    out_dir = config.RESULTS_DIR / "routing"
    out_dir.mkdir(parents=True, exist_ok=True)
    table = []
    for kind in a.routers:
        tracer = Tracer(enabled=False)
        router = make_router(kind, tracer, model=a.api_model, effort=a.api_effort)
        name = router.label if kind != "api" else f"api:{a.api_model}@{a.api_effort}"
        print(f"== {name}: {len(cases)} cases")
        res = evaluate(router, tracer, cases)
        safe = name.replace(":", "_").replace("@", "_").replace("/", "_")
        (out_dir / f"{safe}.json").write_text(json.dumps({"router": name, **res}, ensure_ascii=False, indent=1),
                                              encoding="utf-8")
        s = res["summary"]
        table.append((name, s))
        print(json.dumps({k: v for k, v in s.items() if k != "by_kind"}, ensure_ascii=False))

    print("\n| Router | Routing acc | Intent acc | Acc (eval set) | Acc (test) | JSON valid | Latency mean (ms) | p95 | $/1k |")
    print("|---|---|---|---|---|---|---|---|---|")
    for name, s in table:
        print(f"| {name} | {s['routing_acc']} | {s['intent_acc']} | {s['routing_acc_eval_set']} | {s['routing_acc_test']} "
              f"| {s['json_valid']} | {s['latency_ms_mean']} | {s['latency_ms_p95']} | {s['cost_per_1k_usd']} |")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/a39/training/gen_data.py
"""Sinh dữ liệu SFT cho Router (Agent Routing + Planning + State-aware) — không tốn tiền API.

Mỗi mẫu = (STATE, tin nhắn khách) -> RoutingDecision JSON. Nhãn sinh bằng luật xác định từ template,
nên đúng tuyệt đối với quy tắc trong ROUTER_SYSTEM. Các nhóm mẫu:

  single     : 1 intent, có / không có mã đơn trong câu
  missing    : intent cần order_id nhưng STATE + câu đều không có -> missing_info
  pending    : STATE có pending_intent, khách chỉ đưa mã đơn -> tiếp tục intent cũ
  in_state   : mã đơn đã có trong STATE (lượt trước) -> KHÔNG được hỏi lại (state-aware)
  follow_up  : câu nối tiếp ngắn ("nó còn bảo hành không?") hiểu theo last_agents / entities
  switch     : đang pending nhưng khách hỏi chuyện khác -> bỏ pending
  multi      : 2 agent trong một câu
  chitchat / out_of_scope

Mã đơn trong dữ liệu train nằm ngoài DH1001–DH1012 (dùng cho eval) để tránh leak.
Tuỳ chọn --paraphrase N: dùng Claude (Haiku 5.5, rẻ) viết lại câu khách cho đa dạng, giữ nguyên nhãn.

    python -m training.gen_data --n 3000
    python -m training.gen_data --n 3000 --paraphrase 400
"""
from __future__ import annotations

import argparse
import json
import random
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import config  # noqa: E402
from support.router import NEEDS_ORDER, ROUTER_SYSTEM, build_router_user, normalize_decision  # noqa: E402
from support.state import ConversationState, extract_slots, strip_accents  # noqa: E402

PRODUCTS = json.loads((config.DATA_DIR / "products.json").read_text(encoding="utf-8"))
PNAMES = [p["name"] for p in PRODUCTS] + ["tai nghe SoundMax", "laptop Nova", "Lumi X12", "Pulse Watch",
                                          "bàn phím K75", "chuột M3", "màn hình 27 inch", "Nova Tab"]
CATS = ["tai nghe", "laptop", "điện thoại", "đồng hồ thông minh", "máy tính bảng", "màn hình", "chuột", "bàn phím", "loa"]
NEEDS = ["học online", "chơi game", "làm đồ hoạ", "chạy bộ", "đi du lịch", "văn phòng", "dựng phim", "nghe nhạc"]
BUDGETS = ["2 triệu", "5 triệu", "10 triệu", "15tr", "20 triệu", "25 triệu", "30tr", "dưới 1 triệu"]
FEATURES = ["sạc nhanh", "chống nước", "NFC", "bluetooth 5.3", "kết nối 2 thiết bị", "nâng cấp RAM", "màn OLED", "GPS"]
DEFECTS = ["không lên nguồn", "mất tiếng một bên", "điểm chết", "liệt phím", "sạc không vào", "loạn cảm ứng", "tự tắt nguồn"]

T = {
    "product_inquiry": [
        "tư vấn giúp mình {cat} tầm {budget}", "{pname} giá bao nhiêu vậy?", "{pname} còn hàng không shop",
        "so sánh {pname} với {pname2} giúp mình", "có {cat} nào hợp {need} không?", "{pname} có {feature} không?",
        "mình nên mua {pname} hay {pname2}?", "cách reset {pname} thế nào", "{pname} pin dùng được bao lâu",
        "{cat} nào dưới {budget} đáng mua nhất?", "shop có bán {cat} cho {need} không", "thông số {pname} thế nào",
        "{pname} dùng với {pname2} được không?", "gợi ý {cat} cho {need}, ngân sách {budget}",
    ],
    "order_status": [
        "đơn {oid} tới đâu rồi?", "kiểm tra giúp mình đơn {oid}", "bao giờ mình nhận được đơn {oid}?",
        "mã vận đơn của đơn {oid} là gì", "đơn {oid} giao chưa shop", "sđt {phone} có đơn nào đang giao không?",
        "mình đặt bằng số {phone}, kiểm tra đơn giúp", "đơn {oid} đang ở trạng thái gì",
    ],
    "order_status_noid": [
        "đơn của mình bao giờ tới?", "kiểm tra đơn hàng giúp mình", "sao hàng mãi chưa tới vậy shop",
        "mình muốn biết đơn đang giao tới đâu", "check đơn giúp mình với",
    ],
    "cancel_order": [
        "huỷ đơn {oid} giúp mình", "mình không muốn mua nữa, huỷ đơn {oid}", "cho mình hủy đơn {oid} nhé",
        "đơn {oid} đặt nhầm, huỷ giúp", "mình muốn huỷ đơn {oid} được không",
    ],
    "cancel_order_noid": ["mình muốn huỷ đơn", "hủy đơn giúp mình với", "đặt nhầm rồi, huỷ đơn được không shop"],
    "return_request": [
        "mình muốn trả {pname} trong đơn {oid}", "{pname} đơn {oid} bị {defect}, đổi giúp mình cái mới",
        "đơn {oid} mình đổi ý, muốn trả hàng, còn nguyên seal", "đơn {oid} muốn trả lại vì không vừa ý, đã mở hộp",
        "làm phiếu trả hàng cho đơn {oid} giúp mình", "hàng đơn {oid} bị lỗi, mình muốn hoàn tiền",
        "đơn {oid} trả được không, máy đã kích hoạt rồi",
    ],
    "return_request_noid": [
        "mình muốn trả hàng", "{pname} mới mua bị {defect}, muốn đổi cái khác", "mình muốn hoàn tiền",
        "hàng mới nhận bị lỗi, đổi trả kiểu gì?", "mua về không ưng, trả lại được không shop", "muốn đổi sang màu khác",
    ],
    "warranty_claim": [
        "{pname} đơn {oid} bị {defect}, còn bảo hành không?", "tạo phiếu bảo hành cho đơn {oid} giúp mình",
        "máy đơn {oid} bị {defect}, bảo hành giúp mình", "đơn {oid} còn hạn bảo hành không vậy",
        "{pname} mua ở đơn {oid} hỏng rồi, mang đi bảo hành thế nào",
    ],
    "warranty_claim_noid": [
        "{pname} của mình bị {defect}, bảo hành sao?", "máy mình bị {defect}, còn bảo hành không",
        "mình muốn bảo hành {pname}", "tai nghe mình bị {defect}, gửi bảo hành được không",
    ],
    "policy_question": [
        "chính sách đổi trả thế nào?", "laptop bảo hành bao lâu?", "phí ship bao nhiêu vậy shop", "có trả góp 0% không?",
        "xuất hoá đơn VAT được không", "đơn COD hoàn tiền mất bao lâu", "có giao hàng chủ nhật không",
        "làm rơi vỡ có được bảo hành không", "đổi ý trong bao nhiêu ngày thì được trả", "giao hàng nhanh mất mấy ngày",
        "mua gói bảo hành mở rộng thế nào", "điểm chết màn hình thì chính sách sao", "được kiểm tra hàng trước khi trả tiền không",
        "thanh toán bằng những cách nào", "tai nghe true wireless mở hộp rồi trả được không",
    ],
    "chitchat": ["chào shop", "xin chào", "cảm ơn nhé", "ok cảm ơn bạn nhiều", "hello shop", "shop ơi", "tuyệt, cám ơn"],
    "out_of_scope": [
        "thời tiết hôm nay thế nào", "viết giúp mình đoạn code python sắp xếp mảng", "kể chuyện cười đi",
        "giá vàng hôm nay bao nhiêu", "dịch câu này sang tiếng Anh: tôi yêu Hà Nội", "làm thơ tặng mình đi",
        "ai vô địch world cup 2022", "công thức nấu phở bò",
    ],
}
MULTI = [
    ("đơn {oid} bao giờ tới? với lại {pname} có {feature} không?", ["order", "product"], "order_status"),
    ("{pname} có {feature} không, mua về không ưng thì trả được không?", ["product", "policy"], "product_inquiry"),
    ("đơn {oid} mình đổi địa chỉ nhận được không?", ["order", "policy"], "order_status"),
    ("kiểm tra đơn {oid} và tư vấn thêm {cat} cho {need}", ["order", "product"], "order_status"),
    ("{pname} giá bao nhiêu, có trả góp không?", ["product", "policy"], "product_inquiry"),
    ("đơn {oid} giao chưa, nếu chưa thì phí ship nhanh bao nhiêu?", ["order", "policy"], "order_status"),
]
FOLLOW_ORDER = [  # entities có order_id, last_agents=[order]
    ("nó còn bảo hành không?", "warranty_claim"), ("mình muốn trả lại đơn đó", "return_request"),
    ("huỷ luôn đơn đó giúp mình", "cancel_order"), ("khi nào giao tới vậy", "order_status"),
    ("sản phẩm đó bị {defect}, làm phiếu bảo hành giúp", "warranty_claim"), ("trả hàng được không?", "return_request"),
]
FOLLOW_PRODUCT = [  # entities có product, last_agents=[product]
    "máy đó có {feature} không?", "còn hàng không?", "nó nặng bao nhiêu?", "có màu khác không", "so với {pname2} thì sao?",
]
PENDING_REPLY = ["{oid}", "mã đơn {oid}", "đơn {oid} nhé", "của mình là {oid}", "mã là {oid} shop ơi", "{oid} ạ"]

ASK = {"order_status": "Bạn vui lòng cho mình mã đơn hàng (dạng DHxxxx) hoặc số điện thoại đặt hàng để mình kiểm tra nhé."}
ASK_DEFAULT = "Bạn vui lòng cho mình mã đơn hàng (dạng DHxxxx) để mình hỗ trợ nhé."
CHAT_REPLY = ["Chào bạn, TechViet có thể giúp gì cho bạn ạ?", "Dạ không có gì, bạn cần hỗ trợ thêm gì cứ nhắn mình nhé!"]
OOS_REPLY = "Xin lỗi, mình chỉ hỗ trợ các vấn đề về sản phẩm, đơn hàng, đổi trả và bảo hành của TechViet."
AGENT_OF = {"product_inquiry": "product", "order_status": "order", "cancel_order": "order",
            "return_request": "policy", "warranty_claim": "policy", "policy_question": "policy"}
TASK = {
    ("product_inquiry", "product"): "Tư vấn / trả lời câu hỏi sản phẩm{p}",
    ("order_status", "order"): "Tra cứu trạng thái đơn{o}",
    ("cancel_order", "order"): "Huỷ đơn{o} theo yêu cầu của khách",
    ("return_request", "policy"): "Kiểm tra điều kiện đổi trả đơn{o} và xử lý yêu cầu",
    ("warranty_claim", "policy"): "Kiểm tra bảo hành đơn{o} và xử lý yêu cầu",
    ("policy_question", "policy"): "Trả lời câu hỏi chính sách dựa trên tài liệu",
}


def rand_oid(rng) -> str:
    return f"DH{rng.randint(1013, 9999)}"


def rand_phone(rng) -> str:
    return "0" + rng.choice("35789") + "".join(rng.choice("0123456789") for _ in range(8))


def fill(tpl: str, rng, slots: dict) -> str:
    vals = {"oid": slots.get("order_id") or rand_oid(rng), "phone": slots.get("phone") or rand_phone(rng),
            "pname": slots.get("product") or rng.choice(PNAMES), "pname2": rng.choice(PNAMES), "cat": rng.choice(CATS),
            "need": rng.choice(NEEDS), "budget": rng.choice(BUDGETS), "feature": rng.choice(FEATURES),
            "defect": rng.choice(DEFECTS)}
    text = tpl.format(**vals)
    if "{oid}" in tpl:
        slots["order_id"] = vals["oid"]
    if "{phone}" in tpl:
        slots["phone"] = vals["phone"]
    if "{pname}" in tpl:
        slots["product"] = vals["pname"]
    return text


def noisy(text: str, rng) -> str:
    """Khách hay gõ không dấu / viết hoa đầu câu."""
    r = rng.random()
    if r < 0.15:
        return strip_accents(text)
    if r < 0.5:
        return text[:1].upper() + text[1:]
    return text


def task_for(intent: str, agent: str, ents: dict) -> str:
    o = f" {ents['order_id']}" if ents.get("order_id") else (f" (SĐT {ents['phone']})" if ents.get("phone") else "")
    p = f" {ents['product']}" if ents.get("product") else ""
    return TASK.get((intent, agent), "Xử lý yêu cầu của khách").format(o=o, p=p)


def label(intent: str, agents: list[str], ents: dict, plan_intents: list[str] | None = None,
          missing: bool = False, reply: str = "") -> dict:
    ents = {k: ents.get(k, "") or "" for k in ("order_id", "phone", "product")}
    if missing:
        return normalize_decision({"intent": intent, "missing_info": ["order_id"], "entities": ents,
                                   "reply": ASK.get(intent, ASK_DEFAULT)})
    plan_intents = plan_intents or [intent] * len(agents)
    plan = [{"agent": a, "task": task_for(pi, a, ents)} for a, pi in zip(agents, plan_intents)]
    return normalize_decision({"intent": intent, "agents": agents, "plan": plan, "entities": ents, "reply": reply})


def _state(**kw) -> ConversationState:
    st = ConversationState()
    for k, v in kw.items():
        setattr(st, k, v)
    return st


def sample(rng) -> tuple[str, ConversationState, str, dict]:
    kind = rng.choices(["single", "missing", "pending", "in_state", "follow_product", "switch", "multi", "chat", "oos"],
                       weights=[30, 10, 12, 12, 6, 4, 14, 6, 6])[0]
    st = ConversationState()
    slots: dict = {}

    if kind == "single":
        intent = rng.choice(list(AGENT_OF))
        msg = fill(rng.choice(T[intent]), rng, slots)
        st.merge(extract_slots(msg))
        return kind, st, msg, label(intent, [AGENT_OF[intent]], {**st.entities, **slots})

    if kind == "missing":
        intent = rng.choice(sorted(NEEDS_ORDER))
        msg = fill(rng.choice(T[f"{intent}_noid"]), rng, slots)
        return kind, st, msg, label(intent, [], {**slots, "order_id": ""}, missing=True)

    if kind == "pending":
        intent = rng.choice(sorted(NEEDS_ORDER))
        prev = fill(rng.choice(T[f"{intent}_noid"]), rng, slots)
        st = _state(pending_intent=intent, pending_missing=["order_id"],
                    history=[{"user": prev, "assistant": ASK.get(intent, ASK_DEFAULT)}], turn=1)
        st.merge({"product": slots.get("product", "")})
        if rng.random() < 0.15:                      # khách vẫn chưa đưa mã đơn
            msg = rng.choice(["mình không nhớ mã đơn", "để mình tìm đã", "mã đơn ở đâu nhỉ"])
            return kind, st, msg, label(intent, [], st.entities, missing=True)
        msg = fill(rng.choice(PENDING_REPLY), rng, slots)
        st.merge(extract_slots(msg))
        return kind, st, msg, label(intent, [AGENT_OF[intent]], st.entities)

    if kind == "in_state":
        oid = rand_oid(rng)
        prev = f"đơn {oid} của mình gồm những gì?"
        st = _state(last_agents=["order"], history=[{"user": prev, "assistant": f"Đơn {oid} gồm {rng.choice(PNAMES)}, đã giao."}], turn=1)
        st.merge({"order_id": oid})
        tpl, intent = rng.choice(FOLLOW_ORDER)
        msg = fill(tpl, rng, slots)
        return kind, st, msg, label(intent, [AGENT_OF[intent]], st.entities)

    if kind == "follow_product":
        pname = rng.choice(PNAMES)
        st = _state(last_agents=["product"], turn=1,
                    history=[{"user": f"tư vấn {pname} giúp mình", "assistant": f"{pname} là lựa chọn tốt..."}])
        st.merge({"product": pname})
        msg = fill(rng.choice(FOLLOW_PRODUCT), rng, {})
        return kind, st, msg, label("product_inquiry", ["product"], st.entities)

    if kind == "switch":
        intent = rng.choice(sorted(NEEDS_ORDER))
        prev = fill(rng.choice(T[f"{intent}_noid"]), rng, {})
        st = _state(pending_intent=intent, pending_missing=["order_id"], turn=1,
                    history=[{"user": prev, "assistant": ASK.get(intent, ASK_DEFAULT)}])
        new = rng.choice(["product_inquiry", "policy_question"])
        msg = fill(rng.choice(T[new]), rng, slots)
        return kind, st, msg, label(new, [AGENT_OF[new]], {**st.entities, **slots})

    if kind == "multi":
        tpl, agents, first = rng.choice(MULTI)
        msg = fill(tpl, rng, slots)
        st.merge(extract_slots(msg))
        second = {"product": "product_inquiry", "policy": "policy_question", "order": "order_status"}[agents[1]]
        return kind, st, msg, label("multi", agents, {**st.entities, **slots}, plan_intents=[first, second])

    if kind == "chat":
        msg = rng.choice(T["chitchat"])
        thanks = any(w in msg for w in ("cảm ơn", "cám ơn"))
        return kind, st, msg, label("chitchat", [], {}, reply=CHAT_REPLY[1 if thanks else 0])

    msg = rng.choice(T["out_of_scope"])
    return kind, st, msg, label("out_of_scope", [], {}, reply=OOS_REPLY)


def to_record(kind: str, st: ConversationState, msg: str, target: dict) -> dict:
    user = build_router_user(st.router_view(), msg)
    return {"kind": kind, "user_message": msg, "state": st.router_view(),
            "prompt": [{"role": "system", "content": ROUTER_SYSTEM}, {"role": "user", "content": user}],
            "completion": [{"role": "assistant", "content": json.dumps(target, ensure_ascii=False)}],
            "target": target}


def paraphrase(records: list[dict], n: int, rng) -> list[dict]:
    """Viết lại tin nhắn khách bằng Claude Haiku 5.5, giữ nguyên mã đơn / SĐT / tên sản phẩm."""
    from support.llm import Claude

    llm = Claude(model="claude-haiku-5-5", effort="low", max_tokens=2000)
    schema = {"type": "object", "properties": {"paraphrases": {"type": "array", "items": {"type": "string"}}},
              "required": ["paraphrases"], "additionalProperties": False}
    out = []
    for r in rng.sample(records, min(n, len(records))):
        res = llm.structured("Viết lại câu của khách hàng theo 2 cách khác, văn nói tự nhiên của người Việt, "
                             "giữ nguyên ý định, mã đơn, số điện thoại, tên sản phẩm. Không thêm thông tin.",
                             r["user_message"], schema)
        for p in res["paraphrases"][:2]:
            if extract_slots(p) != extract_slots(r["user_message"]):
                continue                     # paraphrase làm mất / đổi slot -> bỏ
            out.append({**r, "kind": r["kind"] + "+para", "user_message": p,
                        "prompt": [r["prompt"][0], {"role": "user", "content": build_router_user(r["state"], p)}]})
    print(f"paraphrase: +{len(out)} mẫu")
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=3000, help="số mẫu train")
    ap.add_argument("--val", type=int, default=300)
    ap.add_argument("--test", type=int, default=300)
    ap.add_argument("--paraphrase", type=int, default=0, help="số mẫu train đem paraphrase bằng Claude Haiku")
    ap.add_argument("--seed", type=int, default=39)
    a = ap.parse_args()

    config.TRAIN_DATA_DIR.mkdir(parents=True, exist_ok=True)
    splits = {"train": (a.n, a.seed), "val": (a.val, a.seed + 1), "test": (a.test, a.seed + 2)}
    seen: set[str] = set()
    for split, (n, seed) in splits.items():
        rng = random.Random(seed)
        recs = []
        while len(recs) < n:
            kind, st, msg, target = sample(rng)
            msg = noisy(msg, rng)
            key = st.router_view() + msg
            if key in seen:                         # không trùng giữa các split
                continue
            seen.add(key)
            recs.append(to_record(kind, st, msg, target))
        if split == "train" and a.paraphrase:
            recs += paraphrase(recs, a.paraphrase, rng)
        path = config.TRAIN_DATA_DIR / f"sft_{split}.jsonl"
        with path.open("w", encoding="utf-8") as f:
            for r in recs:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        kinds: dict[str, int] = {}
        for r in recs:
            kinds[r["kind"]] = kinds.get(r["kind"], 0) + 1
        print(f"{split}: {len(recs)} mẫu -> {path}  {kinds}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/a39/training/hf_import.py
"""Import dataset Hugging Face `Tson29/techviet-cs-orchestrator-data` -> định dạng Router của project.

Vì sao cần convert?
  Dataset HF train một orchestrator kiểu *từng bước* (mỗi bước 1 JSON:
  {check, action: call_agent|ask_user|finish, agent, task, message, state_update}),
  còn Router của project quyết định *cả lượt* trong 1 JSON
  {intent, agents, plan, missing_info, entities, reply} với prompt ROUTER_SYSTEM + STATE.
  Train thẳng file `sft/train.jsonl` của HF sẽ ra model in sai schema -> normalize_decision() coi là out_of_scope.

Cách làm: đọc config `conversations` (gen/conv_*.jsonl: 300 hội thoại, mỗi lượt có các bước quyết định),
mô phỏng lại ConversationState đúng như Orchestrator (regex slot -> router -> cập nhật pending / last_agents / history),
rồi gộp các bước của một lượt thành 1 RoutingDecision:
  - các bước call_agent         -> agents + plan (mỗi agent 1 bước, task gộp theo thứ tự)
  - chỉ ask_user hỏi mã đơn      -> missing_info ["order_id"] (+ pending_intent cho lượt sau)
  - finish không gọi agent       -> chitchat (agents [], reply = câu trả lời của trợ lý)
  - ask_user hỏi thứ khác / hỏi mã đơn khi STATE đã có mã -> bỏ (không khớp quy tắc ROUTER_SYSTEM)
Output cùng format với training/gen_data.py (prompt / completion / target / kind / state ...), nên
gen_dpo_pairs.py, sft_train.py, dpo_train.py, eval/routing_eval.py dùng được nguyên trạng.

`dpo/dpo.jsonl` của HF là cặp ở mức *bước* (schema khác) nên không dùng; gen_dpo_pairs.py tự sinh lại các
kiểu lỗi tương đương (ask_known_info ~ state_unaware, guess_missing_info ~ no_ask, wrong_agent ...) + on-policy.

Mã đơn: dataset HF dùng DH1001–DH1016, trùng dải mã của data/eval_set.json -> mặc định đổi sang mã ngẫu nhiên
ngoài dải eval (nhất quán trong từng hội thoại). Tắt bằng --keep-order-ids.

    python -m training.hf_import                         # chỉ dữ liệu HF, ghi đè sft_{train,val,test}.jsonl
    python -m training.gen_data --n 1500 && python -m training.hf_import --mode append   # trộn template + HF
    python -m training.hf_import --local-dir path/to/snapshot   # đã tải sẵn (có thư mục gen/)
"""
from __future__ import annotations

import argparse
import copy
import json
import random
import re
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import config  # noqa: E402
from support.router import NEEDS_ORDER, normalize_decision  # noqa: E402
from support.state import ORDER_RE, ConversationState, extract_slots  # noqa: E402
from training.gen_data import AGENT_OF, to_record  # noqa: E402

REPO = "Tson29/techviet-cs-orchestrator-data"
CONV_FILE_RE = re.compile(r"(?:^|/)gen/conv_[^/]+\.jsonl$")
ORDER_TEXT_RE = re.compile(r"\b[Dd]\s?[Hh]\s?-?(\d{4})\b")
ASK_ORDER_RE = re.compile(r"mã đơn|ma don|DHxxxx|DH\s?\d|\bDH\b|số điện thoại|so dien thoai", re.I)

CHAT_KW = ("chào", "chao", "hello", "hi ", "cảm ơn", "cám ơn", "cam on", "thanks", "ok", "oke", "vâng", "dạ", "tạm biệt")
CANCEL_KW = ("huỷ", "hủy", "huy don", "cancel")
WARRANTY_KW = ("bảo hành", "bao hanh")
RETURN_KW = ("đổi trả", "trả hàng", "trả lại", "hoàn tiền", "đổi sang", "đổi mới", "đổi cái", "đổi hàng", "đổi ý",
             "muốn đổi", "muốn trả", "đổi được", "trả được", "doi tra", "tra hang", "hoan tien")
DEFECT_KW = ("hỏng", "lỗi", "không lên", "điểm chết", "bị ", "hong", "loi")


def _has(text: str, kws) -> bool:
    low = text.lower()
    return any(k in low for k in kws)


def _obj(x):
    """Cột kiểu Json trên HF đôi khi bị lưu thành chuỗi JSON."""
    if isinstance(x, str):
        try:
            return json.loads(x)
        except json.JSONDecodeError:
            return x
    return x


# --------------------------------------------------------------------------- #
def load_conversations(repo: str = REPO, local_dir: str | None = None, revision: str | None = None) -> list[dict]:
    if local_dir:
        root = Path(local_dir)
        files = sorted(root.glob("gen/conv_*.jsonl")) or sorted(root.glob("conv_*.jsonl"))
    else:
        from huggingface_hub import HfApi, hf_hub_download

        names = sorted(f for f in HfApi().list_repo_files(repo, repo_type="dataset", revision=revision)
                       if CONV_FILE_RE.search(f))
        files = [Path(hf_hub_download(repo, f, repo_type="dataset", revision=revision)) for f in names]
    if not files:
        raise SystemExit(f"Không tìm thấy gen/conv_*.jsonl trong {local_dir or repo}")
    convs = []
    for p in files:
        for line in p.open(encoding="utf-8"):
            if line.strip():
                convs.append(json.loads(line))
    print(f"Đọc {len(convs)} hội thoại từ {len(files)} file: {[p.name for p in files]}")
    return convs


def remap_order_ids(conv: dict, rng: random.Random, used: set[str]) -> dict:
    """Đổi mọi mã DHxxxx trong hội thoại sang mã ngoài dải eval (DH1001–DH1099), nhất quán trong hội thoại."""
    mapping: dict[str, str] = {}

    def sub(m: re.Match) -> str:
        num = m.group(1)
        if not ("1000" <= num <= "1099"):
            return f"DH{num}"
        if num not in mapping:
            new = f"DH{rng.randint(1100, 9999)}"
            while new in used:
                new = f"DH{rng.randint(1100, 9999)}"
            used.add(new)
            mapping[num] = new
        return mapping[num]

    return json.loads(ORDER_TEXT_RE.sub(sub, json.dumps(conv, ensure_ascii=False)))


# --------------------------------------------------------------------------- #
def category_intent(cat: str) -> str:
    c = (cat or "").lower()
    for key, intent in (("cancel", "cancel_order"), ("warranty", "warranty_claim"), ("return", "return_request"),
                        ("order", "order_status"), ("product", "product_inquiry"), ("policy", "policy_question"),
                        ("chit", "chitchat"), ("greet", "chitchat"), ("small", "chitchat"),
                        ("oos", "out_of_scope"), ("out_of_scope", "out_of_scope")):
        if key in c:
            return intent
    return ""


def infer_intent(agent: str, text: str, has_order: bool) -> str:
    if agent == "product":
        return "product_inquiry"
    if agent == "order":
        return "cancel_order" if _has(text, CANCEL_KW) else "order_status"
    if _has(text, WARRANTY_KW):
        return "warranty_claim" if has_order else "policy_question"
    if _has(text, RETURN_KW):
        return "return_request" if has_order else "policy_question"
    return "policy_question"


def infer_need_order(text: str) -> str:
    """Intent cần mã đơn khi khách chưa đưa mã (lượt ask_user)."""
    if _has(text, CANCEL_KW):
        return "cancel_order"
    if _has(text, RETURN_KW):
        return "return_request"
    if _has(text, WARRANTY_KW) or _has(text, DEFECT_KW):
        return "warranty_claim"
    return "order_status"


def turn_entities(st: ConversationState, decisions: list[dict]) -> dict:
    ents = dict(st.entities)
    for d in decisions:
        su = _obj(d.get("state_update")) or {}
        if not isinstance(su, dict):
            continue
        if su.get("order_id") and (m := ORDER_RE.search(str(su["order_id"]))):
            ents["order_id"] = f"DH{m.group(1)}"
        if su.get("product"):
            ents["product"] = str(su["product"])
    return ents


def kind_of(view: ConversationState, user: str, t: dict) -> str:
    if t["missing_info"]:
        return "pending" if view.pending_intent else "missing"
    if not t["agents"]:
        return "chat" if t["intent"] == "chitchat" else "oos"
    if len(t["agents"]) > 1:
        return "multi"
    if view.pending_intent:
        return "pending" if t["intent"] == view.pending_intent else "switch"
    if t["intent"] in NEEDS_ORDER and view.entities.get("order_id") and not ORDER_RE.search(user):
        return "in_state"
    if t["agents"] == ["product"] and view.last_agents == ["product"] and view.history:
        return "follow_product"
    return "single"


def convert_turn(view: ConversationState, user: str, decisions: list[dict], assistant: str,
                 cat_intent: str) -> tuple[dict | None, str, bool]:
    """-> (target | None nếu bỏ, lý do, có ask_user sau khi đã gọi agent)."""
    ents = turn_entities(view, decisions)
    has_order = bool(ents.get("order_id") or ents.get("phone"))
    calls = [d for d in decisions if d.get("action") == "call_agent" and d.get("agent") in config.AGENTS]
    asks = [d for d in decisions if d.get("action") == "ask_user"]

    if calls:
        tasks: dict[str, list[str]] = {}
        for d in calls:
            tasks.setdefault(d["agent"], [])
            if d.get("task") and d["task"] not in tasks[d["agent"]]:
                tasks[d["agent"]].append(d["task"])
        agents = list(tasks)
        text = user + " " + " ".join(" ".join(v) for v in tasks.values())
        if len(agents) > 1:
            intent = "multi"
        elif view.pending_intent and AGENT_OF.get(view.pending_intent) == agents[0]:
            intent = view.pending_intent                                   # quy tắc 2: tiếp tục pending
        elif AGENT_OF.get(cat_intent) == agents[0] and (cat_intent not in NEEDS_ORDER or has_order):
            intent = cat_intent
        else:
            intent = infer_intent(agents[0], text, has_order)
        plan = [{"agent": a, "task": "; ".join(ts)} for a, ts in tasks.items()]
        target = normalize_decision({"intent": intent, "agents": agents, "plan": plan, "entities": ents})
        return target, "ok", bool(asks)

    if asks:
        msg = asks[0].get("message") or ""
        if not ASK_ORDER_RE.search(msg):
            return None, "ask_other", False                   # schema router chỉ có missing order_id
        if has_order:
            return None, "ask_with_order", False              # vd. mã sai -> ngoài quy tắc 1, để agent xử lý
        if view.pending_intent:
            intent = view.pending_intent
        elif cat_intent in NEEDS_ORDER:
            intent = cat_intent
        else:
            intent = infer_need_order(user)
        target = normalize_decision({"intent": intent, "missing_info": ["order_id"], "entities": ents,
                                     "reply": msg})
        return target, "ok", False

    if _has(user, CHAT_KW) or cat_intent == "chitchat":
        intent = "chitchat"
    elif cat_intent == "out_of_scope":
        intent = "out_of_scope"
    else:
        return None, "finish_no_agent", False                # trả lời từ lịch sử: không có nhãn tương ứng
    target = normalize_decision({"intent": intent, "entities": ents, "reply": assistant})
    return target, "ok", False


def convert_conversation(conv: dict, stats: dict) -> list[dict]:
    st = ConversationState()
    cat_intent = category_intent(conv.get("category", ""))
    recs = []
    for i, turn in enumerate(_obj(conv.get("turns")) or []):
        turn = _obj(turn)
        user = str(turn.get("user") or "").strip()
        if not user:
            continue
        steps = [_obj(s) for s in (_obj(turn.get("steps")) or [])]
        decisions = [d for d in (_obj(s.get("decision")) if isinstance(s, dict) else None for s in steps)
                     if isinstance(d, dict)]
        assistant = str(turn.get("assistant") or "")

        st.merge(extract_slots(user))                    # giống Orchestrator.chat(): regex trước router
        view = copy.deepcopy(st)
        target, why, asked_after = convert_turn(view, user, decisions, assistant, cat_intent)
        stats[why] = stats.get(why, 0) + 1
        if target is not None:
            rec = to_record(kind_of(view, user, target), view, user, target)
            rec.update(source="hf", conv_id=conv.get("id", ""), turn_index=i)
            recs.append(rec)
            st.merge(target["entities"])
            if target["missing_info"]:
                st.pending_intent, st.pending_missing = target["intent"], ["order_id"]
            elif target["agents"] and asked_after:
                st.pending_intent, st.pending_missing = target["intent"], []   # agent trả input-required
            else:
                st.pending_intent, st.pending_missing = "", []
            st.last_agents = target["agents"] or st.last_agents
        else:
            st.merge(turn_entities(st, decisions))
        st.add_turn(user, assistant)
    return recs


# --------------------------------------------------------------------------- #
def write_split(path: Path, recs: list[dict], append: bool) -> int:
    seen = set()
    old = []
    if append and path.exists():
        old = [json.loads(l) for l in path.open(encoding="utf-8") if l.strip()]
        seen = {r["state"] + r["user_message"] for r in old}
    new = [r for r in recs if (r["state"] + r["user_message"]) not in seen]
    with path.open("w", encoding="utf-8") as f:
        for r in old + new:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return len(old) + len(new)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--repo", default=REPO)
    ap.add_argument("--revision", default=None, help="commit / branch trên HF (để tái lập)")
    ap.add_argument("--local-dir", default=None, help="thư mục snapshot đã tải sẵn (chứa gen/conv_*.jsonl)")
    ap.add_argument("--mode", choices=["replace", "append"], default="replace",
                    help="replace: chỉ dữ liệu HF; append: nối vào sft_*.jsonl do gen_data.py sinh")
    ap.add_argument("--val-frac", type=float, default=0.1)
    ap.add_argument("--test-frac", type=float, default=0.1)
    ap.add_argument("--keep-order-ids", action="store_true", help="giữ mã DH10xx gốc (có thể trùng eval set)")
    ap.add_argument("--seed", type=int, default=39)
    a = ap.parse_args()

    rng = random.Random(a.seed)
    convs = load_conversations(a.repo, a.local_dir, a.revision)
    if not a.keep_order_ids:
        used: set[str] = set()
        convs = [remap_order_ids(c, rng, used) for c in convs]

    ids = sorted({c.get("id", str(i)) for i, c in enumerate(convs)})
    rng.shuffle(ids)                                       # chia theo hội thoại -> không lộ lượt giữa các split
    n_val, n_test = int(len(ids) * a.val_frac), int(len(ids) * a.test_frac)
    split_of = {cid: "val" for cid in ids[:n_val]}
    split_of.update({cid: "test" for cid in ids[n_val:n_val + n_test]})

    stats: dict[str, int] = {}
    by_split: dict[str, list[dict]] = {"train": [], "val": [], "test": []}
    for i, c in enumerate(convs):
        by_split[split_of.get(c.get("id", str(i)), "train")] += convert_conversation(c, stats)

    config.TRAIN_DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Lượt: {stats}  (chỉ 'ok' được giữ)")
    for split, recs in by_split.items():
        raw = config.TRAIN_DATA_DIR / f"hf_sft_{split}.jsonl"          # bản riêng để xem lại
        write_split(raw, recs, append=False)
        total = write_split(config.TRAIN_DATA_DIR / f"sft_{split}.jsonl", recs, append=a.mode == "append")
        kinds: dict[str, int] = {}
        for r in recs:
            kinds[r["kind"]] = kinds.get(r["kind"], 0) + 1
        print(f"{split}: +{len(recs)} mẫu HF -> sft_{split}.jsonl (tổng {total})  {kinds}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/a39/training/gen_dpo_pairs.py
"""Sinh cặp preference (prompt, chosen, rejected) cho DPO, nhắm đúng các lỗi điều phối hay gặp.

chosen  = quyết định đúng (nhãn SFT).
rejected lấy từ 2 nguồn:
  1. Nhiễu có chủ đích theo loại mẫu (luôn có, không cần GPU):
     - state_unaware   : mã đơn đã có trong STATE mà vẫn hỏi lại khách
     - no_ask          : thiếu mã đơn mà vẫn route thẳng tới agent (đoán / bịa)
     - ignore_pending  : khách đưa mã đơn để tiếp tục đổi trả, router lại coi là tra đơn
     - drop_agent      : câu multi chỉ route 1 agent
     - extra_agent     : gọi thêm agent không cần (tốn chi phí)
     - wrong_agent     : nhầm product <-> policy <-> order
     - chat_routed     : chào hỏi / ngoài phạm vi mà vẫn gọi agent
     - bad_format      : trả lời văn xuôi thay vì JSON
  2. On-policy (tuỳ chọn --sft-adapter, cần GPU): sample từ model SFT, output nào sai so với nhãn
     thì làm rejected. Đây là lỗi model thực sự mắc, DPO sửa trúng nhất.

    python -m training.gen_dpo_pairs
    python -m training.gen_dpo_pairs --sft-adapter training/checkpoints/sft --onpolicy-n 800
"""
from __future__ import annotations

import argparse
import copy
import json
import random
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import config  # noqa: E402
from eval.metrics import routing_correct  # noqa: E402
from support.router import NEEDS_ORDER, normalize_decision, parse_json_obj  # noqa: E402

CONFUSE = {"policy": ["product", "order"], "product": ["policy"], "order": ["policy"]}
ASK = "Bạn vui lòng cho mình mã đơn hàng (dạng DHxxxx) để mình hỗ trợ nhé."


def _dump(d: dict) -> str:
    return json.dumps(d, ensure_ascii=False)


def _with_agents(t: dict, agents: list[str]) -> dict:
    r = copy.deepcopy(t)
    r["agents"] = agents
    r["plan"] = [{"agent": a, "task": next((s["task"] for s in t["plan"] if s["agent"] == a), "Xử lý yêu cầu của khách")}
                 for a in agents]
    r["missing_info"], r["reply"] = [], ""
    return r


def perturb(rec: dict, rng) -> list[tuple[str, str]]:
    t, kind = rec["target"], rec["kind"].replace("+para", "")
    out: list[tuple[str, str]] = []
    if t["missing_info"]:
        agent = {"order_status": "order", "cancel_order": "order"}.get(t["intent"], "policy")
        out.append(("no_ask", _dump(_with_agents(t, [agent]))))
    elif t["intent"] in ("chitchat", "out_of_scope"):
        out.append(("chat_routed", _dump(_with_agents({**t, "intent": "product_inquiry"}, ["product"]))))
    else:
        if kind in ("in_state", "pending") and t["intent"] in NEEDS_ORDER:
            r = copy.deepcopy(t)
            r.update(agents=[], plan=[], missing_info=["order_id"], reply=ASK)
            r["entities"]["order_id"] = ""
            out.append(("state_unaware", _dump(r)))
        if kind == "pending" and t["intent"] != "order_status":
            r = _with_agents({**t, "intent": "order_status"}, ["order"])
            out.append(("ignore_pending", _dump(r)))
        if len(t["agents"]) > 1:
            out.append(("drop_agent", _dump(_with_agents(t, t["agents"][:1]))))
        else:
            a = t["agents"][0]
            wrong = rng.choice(CONFUSE[a])
            out.append(("wrong_agent", _dump(_with_agents(t, [wrong]))))
            if rng.random() < 0.4:
                out.append(("extra_agent", _dump(_with_agents({**t, "intent": "multi"}, [a, wrong]))))
    if rng.random() < 0.08:
        agents = ", ".join(t["agents"]) or "không agent nào"
        out.append(("bad_format", f"Mình sẽ chuyển yêu cầu này cho {agents} xử lý nhé."))
    return out


def onpolicy_rejections(records: list[dict], adapter: str, n: int, k: int, rng,
                        base_model: str = config.SMALL_BASE_MODEL) -> list[dict]:
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(adapter)
    base = AutoModelForCausalLM.from_pretrained(base_model, torch_dtype=torch.float16, device_map="auto")
    model = PeftModel.from_pretrained(base, adapter).eval()
    pairs = []
    for rec in rng.sample(records, min(n, len(records))):
        enc = tok.apply_chat_template(rec["prompt"], add_generation_prompt=True, return_tensors="pt", return_dict=True)
        ids = enc["input_ids"].to(model.device)
        with torch.no_grad():
            outs = model.generate(ids, max_new_tokens=320, do_sample=True, temperature=0.9, top_p=0.95,
                                  num_return_sequences=k, pad_token_id=tok.eos_token_id)
        seen = set()
        for o in outs:
            text = tok.decode(o[ids.shape[1]:], skip_special_tokens=True).strip()
            raw = parse_json_obj(text)
            t = rec["target"]
            ok = raw is not None and routing_correct(normalize_decision(raw), t["agents"], bool(t["missing_info"]),
                                                     ) and normalize_decision(raw)["intent"] == t["intent"]
            if not ok and text not in seen:
                seen.add(text)
                pairs.append({"prompt": rec["prompt"], "chosen": rec["completion"],
                              "rejected": [{"role": "assistant", "content": text}], "source": "onpolicy"})
    return pairs


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--max-per-prompt", type=int, default=2)
    ap.add_argument("--sft-adapter", default=None, help="bật on-policy negatives từ model SFT (cần GPU)")
    ap.add_argument("--base", default=config.SMALL_BASE_MODEL, help="base model của adapter SFT (phải khớp lúc train)")
    ap.add_argument("--onpolicy-n", type=int, default=600)
    ap.add_argument("--k", type=int, default=4)
    ap.add_argument("--seed", type=int, default=7)
    a = ap.parse_args()
    rng = random.Random(a.seed)

    for split in ("train", "val"):
        recs = [json.loads(l) for l in (config.TRAIN_DATA_DIR / f"sft_{split}.jsonl").open(encoding="utf-8")]
        pairs = []
        for rec in recs:
            cands = perturb(rec, rng)
            rng.shuffle(cands)
            for src, rej in cands[: a.max_per_prompt]:
                pairs.append({"prompt": rec["prompt"], "chosen": rec["completion"],
                              "rejected": [{"role": "assistant", "content": rej}], "source": src})
        if split == "train" and a.sft_adapter:
            pairs += onpolicy_rejections(recs, a.sft_adapter, a.onpolicy_n, a.k, rng, a.base)
        rng.shuffle(pairs)
        path = config.TRAIN_DATA_DIR / f"dpo_{split}.jsonl"
        with path.open("w", encoding="utf-8") as f:
            for p in pairs:
                f.write(json.dumps(p, ensure_ascii=False) + "\n")
        by_src: dict[str, int] = {}
        for p in pairs:
            by_src[p["source"]] = by_src.get(p["source"], 0) + 1
        print(f"{split}: {len(pairs)} cặp -> {path}  {by_src}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/a39/training/sft_train.py
"""SFT Router: fine-tune Small LLM (mặc định Qwen2.5-1.5B-Instruct) bằng LoRA / QLoRA với TRL.

Chạy trên GPU (Colab T4 16GB là đủ cho 1.5B QLoRA; 3B dùng --qlora):
    python -m training.gen_data --n 3000
    python -m training.sft_train --qlora --epochs 2
    python -m training.sft_train --base meta-llama/Llama-3.2-1B-Instruct --out training/checkpoints/sft-llama

Dataset ở dạng conversational prompt-completion (prompt = [system, user], completion = [assistant]),
nên SFTTrainer chỉ tính loss trên phần completion (JSON quyết định), không học lại prompt.
Output: adapter LoRA + tokenizer ở training/checkpoints/sft (vài chục MB).
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import config  # noqa: E402


def load_model(base: str, qlora: bool):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(base)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    dtype = torch.bfloat16 if bf16 else torch.float16
    kw = {"torch_dtype": dtype, "device_map": "auto"}
    if qlora:
        from transformers import BitsAndBytesConfig

        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
    model = AutoModelForCausalLM.from_pretrained(base, **kw)
    if qlora:
        from peft import prepare_model_for_kbit_training

        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    return model, tok, bf16


def lora_config(r: int):
    from peft import LoraConfig

    return LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])


def eval_every(n_train: int, batch: int, grad_accum: int, epochs: float, cap: int = 100) -> int:
    """Số step giữa 2 lần eval/save: ~4 lần mỗi lần train, tối đa `cap` (dữ liệu nhỏ vẫn có eval + best ckpt)."""
    import math

    total = math.ceil(n_train / (batch * grad_accum)) * epochs
    return max(5, min(cap, int(total // 4) or 5))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", default=config.SMALL_BASE_MODEL)
    ap.add_argument("--out", default=str(config.SFT_ADAPTER))
    ap.add_argument("--qlora", action="store_true", help="load 4-bit (QLoRA); không bật = LoRA trên bf16/fp16")
    ap.add_argument("--epochs", type=float, default=2)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--rank", type=int, default=16)
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--grad-accum", type=int, default=2)
    ap.add_argument("--max-length", type=int, default=1536)
    a = ap.parse_args()

    from datasets import load_dataset
    from trl import SFTConfig, SFTTrainer

    data = load_dataset("json", data_files={"train": str(config.TRAIN_DATA_DIR / "sft_train.jsonl"),
                                            "val": str(config.TRAIN_DATA_DIR / "sft_val.jsonl")})
    data = data.select_columns(["prompt", "completion"])
    model, tok, bf16 = load_model(a.base, a.qlora)
    every = eval_every(len(data["train"]), a.batch, a.grad_accum, a.epochs)
    print(f"train={len(data['train'])} val={len(data['val'])} | eval/save mỗi {every} step")

    args = SFTConfig(
        output_dir=a.out, num_train_epochs=a.epochs, learning_rate=a.lr, lr_scheduler_type="cosine",
        warmup_ratio=0.05, per_device_train_batch_size=a.batch, per_device_eval_batch_size=a.batch,
        gradient_accumulation_steps=a.grad_accum, gradient_checkpointing=True,
        max_length=a.max_length, completion_only_loss=True,
        bf16=bf16, fp16=not bf16, logging_steps=min(10, every), eval_strategy="steps", eval_steps=every,
        save_strategy="steps", save_steps=every, save_total_limit=2, load_best_model_at_end=True,
        metric_for_best_model="eval_loss", report_to="none", seed=39,
    )
    trainer = SFTTrainer(model=model, args=args, train_dataset=data["train"], eval_dataset=data["val"],
                         processing_class=tok, peft_config=lora_config(a.rank))
    trainer.model.print_trainable_parameters()
    trainer.train()
    trainer.save_model(a.out)
    tok.save_pretrained(a.out)
    print(f"Đã lưu adapter SFT -> {a.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/a39/training/dpo_train.py
"""DPO: tối ưu quyết định điều phối của Router sau SFT.

Policy    = model SFT (base + adapter SFT), tiếp tục train chính adapter đó.
Reference = bản sao model SFT, đóng băng (DPO kéo xác suất chosen lên / rejected xuống *so với* SFT).
Với 1.5B ở fp16, 2 bản model ~6GB VRAM -> Colab T4 đủ. Model 3B: thêm --qlora.

    python -m training.gen_dpo_pairs [--sft-adapter training/checkpoints/sft]
    python -m training.dpo_train --beta 0.1 --epochs 1

Output: training/checkpoints/dpo (adapter LoRA, load lên base giống adapter SFT).
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import config  # noqa: E402
from training.sft_train import eval_every, load_model  # noqa: E402


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", default=config.SMALL_BASE_MODEL)
    ap.add_argument("--sft", default=str(config.SFT_ADAPTER))
    ap.add_argument("--out", default=str(config.DPO_ADAPTER))
    ap.add_argument("--qlora", action="store_true")
    ap.add_argument("--beta", type=float, default=0.1)
    ap.add_argument("--epochs", type=float, default=1)
    ap.add_argument("--lr", type=float, default=5e-6)
    ap.add_argument("--batch", type=int, default=4)
    ap.add_argument("--grad-accum", type=int, default=4)
    ap.add_argument("--max-length", type=int, default=1536)
    a = ap.parse_args()

    from datasets import load_dataset
    from peft import PeftModel
    from trl import DPOConfig, DPOTrainer

    data = load_dataset("json", data_files={"train": str(config.TRAIN_DATA_DIR / "dpo_train.jsonl"),
                                            "val": str(config.TRAIN_DATA_DIR / "dpo_val.jsonl")})
    data = data.select_columns(["prompt", "chosen", "rejected"])

    base, tok, bf16 = load_model(a.base, a.qlora)
    model = PeftModel.from_pretrained(base, a.sft, is_trainable=True)
    ref_base, _, _ = load_model(a.base, a.qlora)
    ref_model = PeftModel.from_pretrained(ref_base, a.sft, is_trainable=False).eval()
    every = eval_every(len(data["train"]), a.batch, a.grad_accum, a.epochs)
    print(f"train={len(data['train'])} val={len(data['val'])} | eval/save mỗi {every} step")

    args = DPOConfig(
        output_dir=a.out, beta=a.beta, loss_type="sigmoid", num_train_epochs=a.epochs, learning_rate=a.lr,
        lr_scheduler_type="cosine", warmup_ratio=0.1, per_device_train_batch_size=a.batch,
        per_device_eval_batch_size=a.batch, gradient_accumulation_steps=a.grad_accum, gradient_checkpointing=True,
        max_length=a.max_length, bf16=bf16, fp16=not bf16, logging_steps=min(10, every), eval_strategy="steps",
        eval_steps=every, save_strategy="steps", save_steps=every, save_total_limit=2, report_to="none", seed=39,
    )
    trainer = DPOTrainer(model=model, ref_model=ref_model, args=args, train_dataset=data["train"],
                         eval_dataset=data["val"], processing_class=tok)
    trainer.train()
    trainer.save_model(a.out)
    tok.save_pretrained(a.out)
    # Theo dõi trong log: rewards/chosen tăng, rewards/rejected giảm, rewards/accuracies -> ~1.0
    print(f"Đã lưu adapter DPO -> {a.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/a39/tests/test_training.py
"""Test nhẹ cho phần training (không GPU, không mạng, không API). Chỉ cần các file có trong notebook Colab.

    python -m tests.test_training
"""
from __future__ import annotations

import json
import random
import re
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from support.router import RuleRouter, normalize_decision  # noqa: E402


def test_training_data():
    from training.gen_data import sample, to_record
    from training.gen_dpo_pairs import perturb

    rng = random.Random(0)
    kinds = set()
    for _ in range(500):
        kind, st, msg, target = sample(rng)
        kinds.add(kind)
        rec = to_record(kind, st, msg, target)
        assert json.loads(rec["completion"][0]["content"]) == target
        assert target == normalize_decision(target), target
        if target["missing_info"]:
            assert not target["agents"] and not st.entities.get("order_id")
        if kind == "in_state":
            assert target["agents"] and not target["missing_info"], (msg, target)
        for src, rej in perturb(rec, rng):
            assert rej != rec["completion"][0]["content"], src
    assert {"single", "missing", "pending", "in_state", "multi"} <= kinds, kinds
    print("training data ok")


def test_hf_import():
    """Converter dataset HF (orchestrator từng bước) -> RoutingDecision cả lượt, không cần mạng."""
    from training.hf_import import convert_conversation, remap_order_ids

    def step(action, agent=None, task=None, message=None, order_id=None):
        return {"decision": {"check": {"last_ok": None, "reason": ""}, "action": action, "agent": agent, "task": task,
                             "message": message, "state_update": {"order_id": order_id, "product": None, "issue": None}}}

    conv = {"id": "X-001", "category": "return_create", "turns": [
        {"user": "Màn hình mới mua bị điểm chết, muốn đổi", "assistant": "Bạn cho mình mã đơn nhé",
         "steps": [step("ask_user", message="Bạn cho mình xin mã đơn hàng (DH + 4 số) nhé.")]},
        {"user": "dh1009", "assistant": "Đã tạo phiếu đổi trả",
         "steps": [step("call_agent", "policy", "Kiểm tra điều kiện đổi trả đơn DH1009", order_id="dh1009"),
                   step("call_agent", "policy", "Tạo phiếu đổi trả đơn DH1009"), step("finish")]},
        {"user": "đơn đó giao bằng hãng nào vậy", "assistant": "GHN",
         "steps": [json.dumps(step("call_agent", "order", "Tra hãng vận chuyển đơn DH1009")), step("finish")]},
        {"user": "cảm ơn shop", "assistant": "Không có gì ạ", "steps": [step("finish")]},
    ]}
    stats: dict = {}
    recs = convert_conversation(conv, stats)
    t = [r["target"] for r in recs]
    assert [r["kind"] for r in recs] == ["missing", "pending", "in_state", "chat"], [r["kind"] for r in recs]
    assert t[0]["missing_info"] == ["order_id"] and t[0]["intent"] == "return_request" and not t[0]["agents"]
    assert t[1]["intent"] == "return_request" and t[1]["agents"] == ["policy"] and len(t[1]["plan"]) == 1
    assert t[1]["entities"]["order_id"] == "DH1009"
    assert t[2]["agents"] == ["order"] and t[2]["intent"] == "order_status"
    assert '"order_id": "DH1009"' in recs[2]["state"] and '"pending_intent": ""' in recs[2]["state"]
    assert t[3]["intent"] == "chitchat" and t[3]["reply"] == "Không có gì ạ"
    assert '"pending_intent": "return_request"' in recs[1]["state"]

    mapped = remap_order_ids(conv, random.Random(0), set())
    s = json.dumps(mapped, ensure_ascii=False)
    assert "1009" not in s
    assert re.fullmatch(r"DH\d{4}", mapped["turns"][1]["user"])
    print("hf import ok")


def test_routing_eval_cases():
    """eval_set.json đọc được và router luật chạy hết các lượt (kiểm tra file data đã có đủ)."""
    from eval.routing_eval import evaluate, load_cases
    from support.tracing import Tracer

    cases = load_cases(0)
    assert len(cases) >= 30, len(cases)
    tracer = Tracer(enabled=False)
    res = evaluate(RuleRouter(tracer), tracer, cases)
    assert res["summary"]["errors"] == 0, res["summary"]
    print(f"routing eval ok: {len(cases)} lượt, rule acc = {res['summary']['routing_acc']}")


if __name__ == "__main__":
    test_training_data()
    test_hf_import()
    test_routing_eval_cases()
    print("TRAINING OK")


In [ ]:
%%writefile /content/a39/data/products.json
[
  {"id": "P001", "name": "Tai nghe không dây SoundMax Pro", "category": "tai nghe", "brand": "SoundMax", "price": 2490000, "stock": 34,
   "warranty_months": 12, "specs": {"chống ồn": "ANC chủ động -35dB", "pin": "30 giờ (ANC bật)", "kết nối": "Bluetooth 5.3, multipoint", "trọng lượng": "250g"},
   "tags": ["chống ồn", "bluetooth", "over-ear", "làm việc", "du lịch"]},
  {"id": "P002", "name": "Tai nghe True Wireless SoundMax Buds 2", "category": "tai nghe", "brand": "SoundMax", "price": 1290000, "stock": 0,
   "warranty_months": 12, "specs": {"chống ồn": "ANC -25dB", "pin": "8 giờ + 24 giờ hộp sạc", "chống nước": "IPX5", "kết nối": "Bluetooth 5.3"},
   "tags": ["true wireless", "thể thao", "chống nước", "nhỏ gọn"]},
  {"id": "P003", "name": "Tai nghe gaming Vortex G7", "category": "tai nghe", "brand": "Vortex", "price": 1690000, "stock": 12,
   "warranty_months": 24, "specs": {"âm thanh": "7.1 giả lập", "mic": "khử ồn, tháo rời", "kết nối": "USB-C + 3.5mm + 2.4GHz", "pin": "40 giờ"},
   "tags": ["gaming", "mic", "không dây 2.4GHz"]},
  {"id": "P004", "name": "Laptop Nova Air 14", "category": "laptop", "brand": "Nova", "price": 18990000, "stock": 8,
   "warranty_months": 24, "specs": {"CPU": "Core Ultra 5 125H", "RAM": "16GB LPDDR5", "SSD": "512GB", "màn hình": "14\" 2.8K OLED 90Hz", "pin": "75Wh, ~12 giờ", "trọng lượng": "1.2kg"},
   "tags": ["mỏng nhẹ", "văn phòng", "sinh viên", "OLED"]},
  {"id": "P005", "name": "Laptop Nova Creator 16", "category": "laptop", "brand": "Nova", "price": 32990000, "stock": 3,
   "warranty_months": 24, "specs": {"CPU": "Core Ultra 7 155H", "GPU": "RTX 4060 8GB", "RAM": "32GB", "SSD": "1TB", "màn hình": "16\" 3.2K 120Hz 100% DCI-P3", "trọng lượng": "2.1kg"},
   "tags": ["đồ hoạ", "dựng phim", "gaming", "hiệu năng cao"]},
  {"id": "P006", "name": "Laptop Vortex Strike 15", "category": "laptop", "brand": "Vortex", "price": 24490000, "stock": 15,
   "warranty_months": 24, "specs": {"CPU": "Ryzen 7 8845HS", "GPU": "RTX 4070 8GB", "RAM": "16GB DDR5", "SSD": "1TB", "màn hình": "15.6\" FHD 165Hz", "trọng lượng": "2.3kg"},
   "tags": ["gaming", "165Hz", "hiệu năng cao"]},
  {"id": "P007", "name": "Điện thoại Lumi X12", "category": "điện thoại", "brand": "Lumi", "price": 12990000, "stock": 40,
   "warranty_months": 18, "specs": {"chip": "Snapdragon 8s Gen 3", "RAM": "12GB", "bộ nhớ": "256GB", "camera": "50MP OIS + 12MP góc rộng", "pin": "5000mAh, sạc 67W"},
   "tags": ["chụp ảnh", "pin trâu", "sạc nhanh"]},
  {"id": "P008", "name": "Điện thoại Lumi A5", "category": "điện thoại", "brand": "Lumi", "price": 4490000, "stock": 75,
   "warranty_months": 18, "specs": {"chip": "Helio G99", "RAM": "8GB", "bộ nhớ": "128GB", "pin": "6000mAh", "màn hình": "6.7\" 120Hz"},
   "tags": ["giá rẻ", "pin trâu", "học sinh"]},
  {"id": "P009", "name": "Đồng hồ thông minh Pulse Watch 3", "category": "đồng hồ", "brand": "Pulse", "price": 3290000, "stock": 22,
   "warranty_months": 12, "specs": {"màn hình": "AMOLED 1.43\"", "GPS": "băng tần kép", "pin": "14 ngày", "chống nước": "5ATM", "đo": "nhịp tim, SpO2, giấc ngủ"},
   "tags": ["thể thao", "chạy bộ", "GPS", "sức khoẻ"]},
  {"id": "P010", "name": "Máy tính bảng Nova Tab S", "category": "máy tính bảng", "brand": "Nova", "price": 8990000, "stock": 18,
   "warranty_months": 12, "specs": {"màn hình": "11\" 2.5K 120Hz", "chip": "Dimensity 8300", "RAM": "8GB", "bộ nhớ": "256GB", "bút": "hỗ trợ Nova Pen (bán riêng)"},
   "tags": ["học tập", "vẽ", "giải trí"]},
  {"id": "P011", "name": "Sạc nhanh GaN 65W PowerCube", "category": "phụ kiện", "brand": "PowerCube", "price": 590000, "stock": 120,
   "warranty_months": 6, "specs": {"công suất": "65W", "cổng": "2 USB-C + 1 USB-A", "chuẩn": "PD 3.0, PPS"},
   "tags": ["sạc", "GaN", "laptop", "điện thoại"]},
  {"id": "P012", "name": "Chuột không dây Vortex M3", "category": "phụ kiện", "brand": "Vortex", "price": 450000, "stock": 60,
   "warranty_months": 12, "specs": {"DPI": "26000", "kết nối": "2.4GHz + Bluetooth", "pin": "70 giờ", "trọng lượng": "63g"},
   "tags": ["gaming", "nhẹ", "không dây"]},
  {"id": "P013", "name": "Bàn phím cơ Vortex K75", "category": "phụ kiện", "brand": "Vortex", "price": 1890000, "stock": 9,
   "warranty_months": 12, "specs": {"layout": "75%", "switch": "hot-swap, linear", "kết nối": "USB-C + 2.4GHz + Bluetooth", "đèn": "RGB"},
   "tags": ["bàn phím cơ", "gaming", "hot-swap"]},
  {"id": "P014", "name": "Loa bluetooth SoundMax Go", "category": "loa", "brand": "SoundMax", "price": 990000, "stock": 0,
   "warranty_months": 12, "specs": {"công suất": "20W", "chống nước": "IP67", "pin": "16 giờ"},
   "tags": ["di động", "chống nước", "dã ngoại"]},
  {"id": "P015", "name": "Màn hình Nova View 27 4K", "category": "màn hình", "brand": "Nova", "price": 7490000, "stock": 11,
   "warranty_months": 36, "specs": {"kích thước": "27\"", "độ phân giải": "4K IPS", "màu": "99% sRGB, Delta E<2", "cổng": "USB-C 90W, HDMI 2.1, DP"},
   "tags": ["đồ hoạ", "văn phòng", "USB-C"]}
]


In [ ]:
%%writefile /content/a39/data/eval_set.json
[
  {"id": "S01", "category": "product", "turns": [
    {"user": "Mình cần tai nghe chống ồn để làm việc, tầm dưới 3 triệu", "intent": "product_inquiry", "agents": ["product"], "missing": false}],
   "expected_tools": [{"name": "search_products", "args": {}}],
   "criteria": "Gợi ý SoundMax Pro (P001) giá 2.490.000đ, có ANC, còn hàng. Nếu nhắc SoundMax Buds 2 thì phải nói đang hết hàng. Không gợi ý sản phẩm vượt 3 triệu."},

  {"id": "S02", "category": "product", "turns": [
    {"user": "Laptop Nova Air 14 có nâng cấp RAM được không?", "intent": "product_inquiry", "agents": ["product"], "missing": false}],
   "expected_tools": [{"name": "search_knowledge_base", "args": {}}],
   "criteria": "Nói rõ RAM 16GB hàn chết, không nâng cấp được; có thể nhắc SSD nâng cấp được (1 khe M.2)."},

  {"id": "S03", "category": "product", "turns": [
    {"user": "So sánh giúp mình Nova Creator 16 với Vortex Strike 15, mình dựng phim là chính", "intent": "product_inquiry", "agents": ["product"], "missing": false}],
   "expected_tools": [{"name": "compare_products", "args": {"product_ids": ["P005", "P006"]}}],
   "criteria": "So sánh đúng thông số (Creator 16: RTX 4060, 32GB RAM, màn 3.2K 100% DCI-P3, 32.990.000đ; Strike 15: RTX 4070, 16GB, FHD 165Hz, 24.490.000đ) và khuyên Creator 16 cho dựng phim hoặc giải thích hợp lý."},

  {"id": "S04", "category": "product", "turns": [
    {"user": "Loa SoundMax Go còn hàng không shop?", "intent": "product_inquiry", "agents": ["product"], "missing": false}],
   "expected_tools": [{"name": "search_products", "args": {}}],
   "criteria": "Báo loa SoundMax Go đang hết hàng. Điểm cộng nếu nói có thể đăng ký nhận thông báo / dự kiến 7–10 ngày về hàng."},

  {"id": "S05", "category": "order", "turns": [
    {"user": "Đơn DH1002 của mình giao tới đâu rồi?", "intent": "order_status", "agents": ["order"], "missing": false}],
   "expected_tools": [{"name": "lookup_order", "args": {"order_id": "DH1002"}}],
   "criteria": "Đơn đang giao (shipping), dự kiến 2026-10-30, mã vận đơn GHN-88812345."},

  {"id": "S06", "category": "order", "turns": [
    {"user": "Mình không nhớ mã đơn, số điện thoại 0901234567, kiểm tra đơn giúp mình", "intent": "order_status", "agents": ["order"], "missing": false}],
   "expected_tools": [{"name": "list_orders_by_phone", "args": {"phone": "0901234567"}}],
   "criteria": "Liệt kê 2 đơn DH1001 (tai nghe SoundMax Pro) và DH1004 (Lumi X12), cả hai đã giao."},

  {"id": "S07", "category": "state", "turns": [
    {"user": "Kiểm tra đơn hàng giúp mình với", "intent": "order_status", "agents": [], "missing": true},
    {"user": "DH1003", "intent": "order_status", "agents": ["order"], "missing": false}],
   "expected_tools": [{"name": "lookup_order", "args": {"order_id": "DH1003"}}],
   "criteria": "Lượt 1 hỏi mã đơn. Lượt 2 báo đơn DH1003 đang xử lý (processing), dự kiến giao 2026-11-02, sản phẩm Pulse Watch 3."},

  {"id": "S08", "category": "order", "turns": [
    {"user": "Mình muốn huỷ đơn DH1003, không cần nữa", "intent": "cancel_order", "agents": ["order"], "missing": false}],
   "expected_tools": [{"name": "cancel_order", "args": {"order_id": "DH1003"}}],
   "criteria": "Xác nhận đã huỷ đơn DH1003 thành công và tiền (nếu đã chuyển khoản) hoàn trong 3–5 ngày làm việc."},

  {"id": "S09", "category": "order", "turns": [
    {"user": "Huỷ đơn DH1002 giúp mình", "intent": "cancel_order", "agents": ["order"], "missing": false}],
   "expected_tools": [{"name": "cancel_order", "args": {"order_id": "DH1002"}}],
   "criteria": "Giải thích không huỷ được vì đơn đang giao (shipping); hướng dẫn từ chối nhận hàng khi shipper giao. Không được nói là đã huỷ."},

  {"id": "S10", "category": "policy", "turns": [
    {"user": "Chính sách đổi trả bên mình thế nào nếu mình đổi ý không muốn dùng nữa?", "intent": "policy_question", "agents": ["policy"], "missing": false}],
   "expected_tools": [{"name": "search_policy", "args": {}}],
   "criteria": "Nêu: trong 15 ngày kể từ ngày nhận; còn seal hoàn 100%; đã mở hộp chưa kích hoạt hoàn 90%; điện thoại/laptop/tablet/đồng hồ đã kích hoạt không áp dụng."},

  {"id": "S11", "category": "policy", "turns": [
    {"user": "Trả góp 0% áp dụng cho đơn từ bao nhiêu tiền?", "intent": "policy_question", "agents": ["policy"], "missing": false}],
   "expected_tools": [{"name": "search_policy", "args": {}}],
   "criteria": "Đơn từ 3 triệu đồng, kỳ hạn 6 hoặc 12 tháng, qua thẻ tín dụng hoặc công ty tài chính."},

  {"id": "S12", "category": "policy", "turns": [
    {"user": "Đơn thanh toán COD thì hoàn tiền mất bao lâu?", "intent": "policy_question", "agents": ["policy"], "missing": false}],
   "expected_tools": [{"name": "search_policy", "args": {}}],
   "criteria": "Hoàn bằng chuyển khoản trong 5–7 ngày làm việc, cần số tài khoản chính chủ."},

  {"id": "S13", "category": "state", "turns": [
    {"user": "Màn hình mình mới mua về bị điểm chết, muốn đổi cái khác", "intent": "return_request", "agents": [], "missing": true},
    {"user": "Mã đơn DH1009, đổi giúp mình cái mới nhé", "intent": "return_request", "agents": ["policy"], "missing": false}],
   "expected_tools": [{"name": "check_return_eligibility", "args": {"order_id": "DH1009", "reason": "defect"}},
                      {"name": "create_return_request", "args": {"order_id": "DH1009", "product_id": "P015", "reason": "defect"}}],
   "criteria": "Lượt 1 hỏi mã đơn. Lượt 2: lỗi NSX trong 30 ngày được đổi mới miễn phí; đã tạo phiếu RMA (nêu mã RMA-...) và bước tiếp theo."},

  {"id": "S14", "category": "policy", "turns": [
    {"user": "Mình muốn trả tai nghe đơn DH1001 vì không thích, hộp vẫn còn seal", "intent": "return_request", "agents": ["policy"], "missing": false}],
   "expected_tools": [{"name": "check_return_eligibility", "args": {"order_id": "DH1001", "reason": "change_of_mind"}}],
   "criteria": "Không đủ điều kiện đổi ý vì đã quá 15 ngày (nhận 2026-10-14, hôm nay là ngày thứ 16). Không được tạo phiếu đổi trả."},

  {"id": "S15", "category": "policy", "turns": [
    {"user": "Đơn DH1005 mình muốn trả tai nghe Buds 2, đã mở hộp dùng thử nhưng không thích", "intent": "return_request", "agents": ["policy"], "missing": false}],
   "expected_tools": [{"name": "check_return_eligibility", "args": {"order_id": "DH1005", "product_id": "P002", "reason": "change_of_mind", "condition": "opened_unactivated"}}],
   "criteria": "Không áp dụng đổi ý vì tai nghe true wireless đã mở hộp (lý do vệ sinh). Không tạo phiếu."},

  {"id": "S16", "category": "policy", "turns": [
    {"user": "Đơn DH1005 cho mình trả con chuột Vortex M3, chưa bóc seal, làm phiếu trả hàng luôn giúp mình", "intent": "return_request", "agents": ["policy"], "missing": false}],
   "expected_tools": [{"name": "check_return_eligibility", "args": {"order_id": "DH1005", "product_id": "P012", "reason": "change_of_mind", "condition": "sealed"}},
                      {"name": "create_return_request", "args": {"order_id": "DH1005", "product_id": "P012", "reason": "change_of_mind", "condition": "sealed"}}],
   "criteria": "Tạo phiếu RMA thành công, hoàn 100% = 450.000đ, đơn COD nên hoàn bằng chuyển khoản 5–7 ngày làm việc, nêu mã phiếu."},

  {"id": "S17", "category": "policy", "turns": [
    {"user": "Đơn DH1012 mình mua 2 điện thoại Lumi A5, muốn trả cả hai vì đổi ý. Một máy đã kích hoạt, một máy còn nguyên seal. Được không?", "intent": "return_request", "agents": ["policy"], "missing": false}],
   "expected_tools": [{"name": "check_return_eligibility", "args": {"order_id": "DH1012", "reason": "change_of_mind", "condition": "sealed"}},
                      {"name": "check_return_eligibility", "args": {"order_id": "DH1012", "reason": "change_of_mind", "condition": "activated"}}],
   "criteria": "Máy còn seal: trả được, hoàn 100% (4.490.000đ). Máy đã kích hoạt: không áp dụng đổi ý. Không tự tạo phiếu khi khách chỉ hỏi 'được không'."},

  {"id": "S18", "category": "policy", "turns": [
    {"user": "Laptop Vortex Strike 15 của mình (đơn DH1007) bị liệt bàn phím, còn bảo hành không? Còn thì tạo phiếu bảo hành giúp mình", "intent": "warranty_claim", "agents": ["policy"], "missing": false}],
   "expected_tools": [{"name": "check_warranty", "args": {"order_id": "DH1007"}},
                      {"name": "create_warranty_ticket", "args": {"order_id": "DH1007", "product_id": "P006"}}],
   "criteria": "Còn bảo hành đến 2027-09-04; đã tạo phiếu bảo hành (mã BH-...), thời gian xử lý 7–14 ngày làm việc, miễn phí gửi đi (sản phẩm trên 5 triệu)."},

  {"id": "S19", "category": "policy", "turns": [
    {"user": "Điện thoại Lumi X12 đơn DH1004 mình làm rơi vỡ màn hình, có được bảo hành không?", "intent": "warranty_claim", "agents": ["policy"], "missing": false}],
   "expected_tools": [{"name": "check_warranty", "args": {"order_id": "DH1004"}}, {"name": "search_policy", "args": {}}],
   "criteria": "Máy vẫn trong thời hạn bảo hành nhưng rơi vỡ không thuộc diện bảo hành; có thể sửa chữa có tính phí. Không tạo phiếu bảo hành miễn phí."},

  {"id": "S20", "category": "policy", "turns": [
    {"user": "Bàn phím đơn DH1008 bị liệt mấy phím, mình muốn đổi cái mới", "intent": "return_request", "agents": ["policy"], "missing": false}],
   "expected_tools": [{"name": "check_return_eligibility", "args": {"order_id": "DH1008", "reason": "defect"}}],
   "criteria": "Nhận hàng 2026-10-03 (ngày thứ 27) nên lỗi NSX vẫn trong 30 ngày: được đổi mới cùng model hoặc hoàn 100%, miễn phí vận chuyển."},

  {"id": "S21", "category": "multi", "turns": [
    {"user": "Đơn DH1002 bao giờ tới? Với lại laptop trong đơn đó sạc bằng củ 65W được không?", "intent": "multi", "agents": ["order", "product"], "missing": false}],
   "expected_tools": [{"name": "lookup_order", "args": {"order_id": "DH1002"}}, {"name": "search_knowledge_base", "args": {}}],
   "criteria": "Dự kiến giao 2026-10-30 (đang giao). Laptop Nova Air 14 sạc USB-C PD tối thiểu 65W nên củ 65W dùng được (đơn cũng có sạc GaN 65W PowerCube)."},

  {"id": "S22", "category": "multi", "turns": [
    {"user": "Pulse Watch 3 có đeo đi bơi được không, và nếu mua về không ưng thì trả được không?", "intent": "multi", "agents": ["product", "policy"], "missing": false}],
   "expected_tools": [{"name": "search_knowledge_base", "args": {}}, {"name": "search_policy", "args": {}}],
   "criteria": "5ATM: bơi bể bơi được, không lặn sâu / xông hơi. Đổi ý trong 15 ngày; đồng hồ đã kích hoạt thì không áp dụng đổi ý, còn seal hoàn 100%."},

  {"id": "S23", "category": "state", "turns": [
    {"user": "Đơn DH1011 của mình gồm những gì vậy?", "intent": "order_status", "agents": ["order"], "missing": false},
    {"user": "Nó còn bảo hành không?", "intent": "warranty_claim", "agents": ["policy"], "missing": false}],
   "expected_tools": [{"name": "lookup_order", "args": {"order_id": "DH1011"}}, {"name": "check_warranty", "args": {"order_id": "DH1011"}}],
   "criteria": "Lượt 1: laptop Nova Creator 16, đã giao 2025-12-12. Lượt 2: còn bảo hành đến 2027-12-12 (24 tháng)."},

  {"id": "S24", "category": "state", "turns": [
    {"user": "Tư vấn giúp mình điện thoại pin trâu dưới 5 triệu", "intent": "product_inquiry", "agents": ["product"], "missing": false},
    {"user": "Máy đó có NFC không?", "intent": "product_inquiry", "agents": ["product"], "missing": false}],
   "expected_tools": [{"name": "search_products", "args": {}}, {"name": "search_knowledge_base", "args": {}}],
   "criteria": "Lượt 1 gợi ý Lumi A5 4.490.000đ, pin 6000mAh. Lượt 2 trả lời Lumi A5 không có NFC."},

  {"id": "S25", "category": "chitchat", "turns": [
    {"user": "Chào shop", "intent": "chitchat", "agents": [], "missing": false}],
   "expected_tools": [],
   "criteria": "Chào lại thân thiện và hỏi có thể giúp gì. Không gọi tool."},

  {"id": "S26", "category": "out_of_scope", "turns": [
    {"user": "Viết cho mình một bài thơ về mùa thu Hà Nội", "intent": "out_of_scope", "agents": [], "missing": false}],
   "expected_tools": [],
   "criteria": "Từ chối lịch sự / hướng khách về các dịch vụ của cửa hàng. Không viết bài thơ dài."},

  {"id": "S27", "category": "policy", "turns": [
    {"user": "Mình muốn trả hàng đơn DH1006", "intent": "return_request", "agents": ["policy"], "missing": false}],
   "expected_tools": [{"name": "check_return_eligibility", "args": {"order_id": "DH1006"}}],
   "criteria": "Đơn DH1006 đã bị huỷ (đã hoàn tiền 2026-10-17) nên không có gì để trả hàng."},

  {"id": "S28", "category": "state", "turns": [
    {"user": "Tai nghe gaming của mình bị mất tiếng một bên, bảo hành kiểu gì?", "intent": "warranty_claim", "agents": [], "missing": true},
    {"user": "đơn DH1010 nhé", "intent": "warranty_claim", "agents": ["policy"], "missing": false}],
   "expected_tools": [{"name": "check_warranty", "args": {"order_id": "DH1010"}}],
   "criteria": "Lượt 1 hỏi mã đơn. Lượt 2: đơn DH1010 đã được trả hàng và hoàn tiền (2026-10-02) nên không thể bảo hành theo đơn này; không tạo phiếu."},

  {"id": "S29", "category": "policy", "turns": [
    {"user": "Giao hàng nhanh mất mấy ngày, phí bao nhiêu?", "intent": "policy_question", "agents": ["policy"], "missing": false}],
   "expected_tools": [{"name": "search_policy", "args": {}}],
   "criteria": "1–2 ngày làm việc (nội thành Hà Nội/TP.HCM 4 giờ), phí 50.000đ."},

  {"id": "S30", "category": "multi", "turns": [
    {"user": "Đơn DH1002 mình đổi địa chỉ nhận hàng được không?", "intent": "multi", "agents": ["order", "policy"], "missing": false}],
   "expected_tools": [{"name": "lookup_order", "args": {"order_id": "DH1002"}}, {"name": "search_policy", "args": {}}],
   "criteria": "Không đổi được vì đơn đang giao (shipping); chỉ đổi địa chỉ khi đơn còn processing."},

  {"id": "S31", "category": "product", "turns": [
    {"user": "Laptop gaming nào dưới 25 triệu còn hàng?", "intent": "product_inquiry", "agents": ["product"], "missing": false}],
   "expected_tools": [{"name": "search_products", "args": {}}],
   "criteria": "Gợi ý Vortex Strike 15 giá 24.490.000đ (RTX 4070, 165Hz), còn hàng. Không gợi ý Creator 16 (32.990.000đ) như là dưới 25 triệu."},

  {"id": "S32", "category": "product", "turns": [
    {"user": "Màn hình Nova View 27 cắm một dây USB-C vào Nova Air 14 vừa xuất hình vừa sạc được không?", "intent": "product_inquiry", "agents": ["product"], "missing": false}],
   "expected_tools": [{"name": "search_knowledge_base", "args": {}}],
   "criteria": "Được: cổng USB-C của màn hình cấp nguồn 90W, một dây vừa xuất hình vừa sạc, Nova Air 14 dùng được."}
]


## Cài đặt và kiểm tra

In [ ]:
# 3. Cài thư viện (torch có sẵn trên Colab)
!pip install -q "transformers>=4.45" "peft>=0.12" "accelerate>=0.33" "trl>=0.19" "datasets>=2.20" "bitsandbytes>=0.43" "anthropic>=1.0"
import torch, transformers, trl, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| trl", trl.__version__, "| peft", peft.__version__)
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime -> Change runtime type -> T4 GPU"
print("GPU:", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())

In [ ]:
# 4. (Tuỳ chọn) API key từ Colab Secrets, chỉ cần cho PARAPHRASE > 0 hoặc EVAL_API
key = None
try:
    from google.colab import userdata
    key = userdata.get("ANTHROPIC_API_KEY")
except Exception as e:
    print("Không đọc được secret ANTHROPIC_API_KEY:", type(e).__name__)
if key:
    os.environ["ANTHROPIC_API_KEY"] = key
    print("Đã nạp ANTHROPIC_API_KEY")
elif PARAPHRASE or EVAL_API:
    raise SystemExit("PARAPHRASE / EVAL_API cần ANTHROPIC_API_KEY trong Secrets")
else:
    print("Không dùng API, ổn: các bước train không gọi Claude")

In [ ]:
# 5. Test nhẹ (không GPU, không API): code + nhãn dữ liệu nhất quán trước khi tốn giờ GPU
!python -m tests.test_training

## Dữ liệu

In [ ]:
# 6. Dữ liệu SFT: training/data/sft_{train,val,test}.jsonl
#    - template: training.gen_data (luật, đủ nhóm state-aware)
#    - hf      : training.hf_import convert hội thoại HF (orchestrator từng bước) -> RoutingDecision cả lượt
if DATA_SOURCE == "template":
    !python -m training.gen_data --n {N_TRAIN} --paraphrase {PARAPHRASE}
elif DATA_SOURCE == "hf":
    !python -m training.hf_import --repo {HF_REPO} --mode replace
elif DATA_SOURCE == "hf+template":
    !python -m training.gen_data --n {N_TRAIN} --paraphrase {PARAPHRASE}
    !python -m training.hf_import --repo {HF_REPO} --mode append
else:
    raise ValueError(DATA_SOURCE)
!wc -l training/data/*.jsonl

In [ ]:
# 6b. Xem thử 1 mẫu convert từ HF (prompt router nhìn thấy + nhãn)
import json
p = pathlib.Path("training/data/hf_sft_train.jsonl")
if p.exists():
    r = json.loads(p.open(encoding="utf-8").readline())
    print("kind:", r["kind"], "| conv:", r.get("conv_id"))
    print(r["prompt"][1]["content"])
    print("->", json.dumps(r["target"], ensure_ascii=False, indent=1))

## Train

In [ ]:
# 7. Baseline: Small LLM chưa fine-tune (+ router luật), để có mốc so sánh
!python -m eval.routing_eval --routers rule small --test-limit {TEST_LIMIT}

In [ ]:
# 8. SFT (LoRA / QLoRA) -> training/checkpoints/sft   (theo dõi eval_loss giảm)
qlora = "--qlora" if QLORA else ""
!python -m training.sft_train --base {BASE_MODEL} {qlora} --epochs {SFT_EPOCHS}
!ls -la training/checkpoints/sft

In [ ]:
# 9. Cặp DPO: negatives theo 8 kiểu lỗi điều phối + on-policy (lỗi model SFT thật sự mắc)
#    (dpo/dpo.jsonl của HF là cặp mức "bước", khác schema -> không dùng; script sinh lại các lỗi tương đương)
onp = f"--sft-adapter training/checkpoints/sft --onpolicy-n {ONPOLICY_N}" if ONPOLICY_N else ""
!python -m training.gen_dpo_pairs --base {BASE_MODEL} {onp}
!wc -l training/data/dpo_*.jsonl

In [ ]:
# 10. DPO trên nền SFT (ref = SFT đóng băng) -> training/checkpoints/dpo
#     Theo dõi: rewards/accuracies -> ~1.0, rewards/chosen > rewards/rejected. OOM: thêm --batch 2 --grad-accum 8
import gc; gc.collect(); torch.cuda.empty_cache()
!python -m training.dpo_train --base {BASE_MODEL} {qlora} --beta {DPO_BETA} --epochs {DPO_EPOCHS}
!ls -la training/checkpoints/dpo

## Đánh giá và tải kết quả

In [ ]:
# 11. So sánh router: rule vs base vs SFT vs DPO (+ API nếu bật)
routers = "rule small sft dpo" + (" api" if EVAL_API else "")
!python -m eval.routing_eval --routers {routers} --test-limit {TEST_LIMIT}

In [ ]:
# 12. Bảng accuracy theo nhóm (state-aware: pending / in_state / follow_product / multi ...)
import glob, pandas as pd
rows = {}
for p in sorted(glob.glob("results/routing/*.json")):
    r = json.load(open(p, encoding="utf-8"))
    s = r["summary"]
    row = {k: s.get(k) for k in ("routing_acc", "intent_acc", "routing_acc_eval_set", "routing_acc_test",
                                 "json_valid", "latency_ms_mean", "cost_per_1k_usd")}
    row.update({f"acc:{k}": v.get("acc") for k, v in s.get("by_kind", {}).items()})
    rows[r["router"]] = row
pd.DataFrame(rows).T

In [ ]:
# 13. Đóng gói adapter (bỏ checkpoint trung gian), dữ liệu, kết quả eval rồi tải về
import zipfile, shutil
out = pathlib.Path("/content/a39_router_artifacts.zip")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for pat in ("training/checkpoints/sft/*", "training/checkpoints/dpo/*", "training/data/*", "results/routing/*"):
        for f in glob.glob(pat):
            if os.path.isfile(f):                         # bỏ thư mục checkpoint-XXX (optimizer state, rất nặng)
                z.write(f)
print(out, f"{out.stat().st_size / 1e6:.1f} MB")
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copy(out, "/content/drive/MyDrive/a39_router_artifacts.zip")
    print("Đã chép vào MyDrive/a39_router_artifacts.zip")
from google.colab import files
files.download(str(out))